In [1]:
!pip install scikit-learn plotly pandas numpy

# Cell 1: Import Libraries dan Setup + Save Setup
import pandas as pd
import numpy as np
import os
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

# Machine Learning Libraries
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Visualization Libraries
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create folders for saving components
os.makedirs('flask_components', exist_ok=True)
os.makedirs('flask_components/models', exist_ok=True)
os.makedirs('flask_components/data', exist_ok=True)
os.makedirs('flask_components/visualizations', exist_ok=True)
os.makedirs('flask_components/results', exist_ok=True)

print("✅ Semua library berhasil diimport!")
print("🔧 Setup environment selesai!")
print("📊 Siap untuk analisis prediksi harga bahan pangan dengan Support Vector Regression")
print("💾 Folder untuk Flask components sudah dibuat!")

# Save setup info
setup_info = {
    'libraries': ['pandas', 'numpy', 'scikit-learn', 'plotly'],
    'folders_created': ['flask_components', 'models', 'data', 'visualizations', 'results'],
    'timestamp': pd.Timestamp.now().isoformat(),
    'research_title': 'PENERAPAN METODE SUPPORT VECTOR REGRESSION DALAM MEMPREDIKSI HARGA BAHAN PANGAN DI KABUPATEN LANGKAT'
}

with open('flask_components/setup_info.json', 'w', encoding='utf-8') as f:
    json.dump(setup_info, f, indent=2)

print("\n📋 INFORMASI PENELITIAN:")
print("=" * 80)
print("Judul: PENERAPAN METODE SUPPORT VECTOR REGRESSION")
print("       DALAM MEMPREDIKSI HARGA BAHAN PANGAN DI KABUPATEN LANGKAT")
print("=" * 80)
print("Metode: Support Vector Regression (SVR) dengan kernel RBF")
print("Metrik Evaluasi: MAPE, RMSE, MAE")
print("Target: Prediksi harga bahan pangan (6 kategori beras)")

zsh:1: command not found: pip


✅ Semua library berhasil diimport!
🔧 Setup environment selesai!
📊 Siap untuk analisis prediksi harga bahan pangan dengan Support Vector Regression
💾 Folder untuk Flask components sudah dibuat!

📋 INFORMASI PENELITIAN:
Judul: PENERAPAN METODE SUPPORT VECTOR REGRESSION
       DALAM MEMPREDIKSI HARGA BAHAN PANGAN DI KABUPATEN LANGKAT
Metode: Support Vector Regression (SVR) dengan kernel RBF
Metrik Evaluasi: MAPE, RMSE, MAE
Target: Prediksi harga bahan pangan (6 kategori beras)


In [2]:
# Cell 2: Load dan Eksplorasi Dataset + Merge Data Cuaca + Save Dataset Info

# Load dataset harga beras (wide format: 1 kolom per kategori)
df = pd.read_csv('dataset.csv')

print("📊 INFORMASI DATASET PREDIKSI HARGA BAHAN PANGAN KABUPATEN LANGKAT")
print("=" * 80)
print(f"Judul Penelitian: PENERAPAN METODE SUPPORT VECTOR REGRESSION")
print(f"                  DALAM MEMPREDIKSI HARGA BAHAN PANGAN DI KABUPATEN LANGKAT")
print("=" * 80)
print(f"Jumlah baris: {len(df)}")
print(f"Jumlah kolom: {len(df.columns)}")

# Convert tanggal ke datetime
df['Tanggal'] = pd.to_datetime(df['Tanggal'], format='%m/%d/%Y')
df = df.sort_values('Tanggal').reset_index(drop=True)

# ===== MERGE DATA CUACA (Temperatur & Curah Hujan) =====
print(f"\n🌦️  MENGGABUNGKAN DATA CUACA HARIAN...")
print("=" * 80)

cuaca = pd.read_excel('cuaca_harian.xlsx')
cuaca['Tanggal'] = pd.to_datetime(cuaca['Tanggal'])

print(f"Data cuaca: {len(cuaca)} hari ({cuaca['Tanggal'].min().strftime('%Y-%m-%d')} s/d {cuaca['Tanggal'].max().strftime('%Y-%m-%d')})")
print(f"Kolom cuaca: {cuaca.columns.tolist()}")

df = df.merge(cuaca[['Tanggal', 'Temperatur', 'Curah_Hujan']], on='Tanggal', how='left')

missing_cuaca = df[['Temperatur', 'Curah_Hujan']].isnull().sum().sum()
if missing_cuaca > 0:
    print(f"⚠️ Ditemukan {missing_cuaca} nilai cuaca kosong setelah merge, mengisi dengan interpolasi...")
    df['Temperatur'] = df['Temperatur'].interpolate().ffill().bfill()
    df['Curah_Hujan'] = df['Curah_Hujan'].interpolate().ffill().bfill()
else:
    print("✅ Data cuaca berhasil digabungkan, tidak ada nilai kosong!")

weather_features = ['Temperatur', 'Curah_Hujan']

print(f"\nPeriode data: {df['Tanggal'].min().strftime('%Y-%m-%d')} s/d {df['Tanggal'].max().strftime('%Y-%m-%d')}")
print(f"Total hari: {len(df)} hari")
print(f"Rentang tahun: {df['Tahun'].min()} - {df['Tahun'].max()}")

# TAMPILKAN DATASET DALAM BENTUK DATAFRAME
print(f"\n📋 TABEL 4.2 DATASET PENELITIAN (Harga + Cuaca):")
print("=" * 120)
print("\n10 Data PERTAMA:")
display(df.head(10))

print("\n10 Data TERAKHIR:")
display(df.tail(10))

# Statistik deskriptif untuk semua kategori beras
kategori_beras = [col for col in df.columns if 'Beras' in col]

print(f"\n📊 KATEGORI BERAS YANG DIPREDIKSI:")
print("=" * 80)
for i, kategori in enumerate(kategori_beras, 1):
    mean_price = df[kategori].mean()
    min_price = df[kategori].min()
    max_price = df[kategori].max()
    print(f"{i}. {kategori}")
    print(f"   Mean: Rp {mean_price:,.0f} | Min: Rp {min_price:,.0f} | Max: Rp {max_price:,.0f}")

# Tampilkan statistik deskriptif dalam bentuk tabel
print(f"\n📊 STATISTIK DESKRIPTIF HARGA BERAS:")
print("=" * 120)
desc_stats = df[kategori_beras].describe().round(0)
display(desc_stats)

# Statistik deskriptif data cuaca
print(f"\n🌦️  STATISTIK DESKRIPTIF DATA CUACA:")
print("=" * 120)
weather_desc_stats = df[weather_features].describe().round(2)
display(weather_desc_stats)

# Korelasi cuaca terhadap harga beras (cek relevansi fitur cuaca)
print(f"\n🔍 KORELASI CUACA TERHADAP HARGA BERAS:")
print("=" * 80)
weather_corr = df[weather_features + kategori_beras].corr()[weather_features].loc[kategori_beras]
display(weather_corr.round(3))

# Informasi kolom
print(f"\n📋 INFORMASI KOLOM DATASET:")
print("=" * 80)
info_df = pd.DataFrame({
    'Kolom': df.columns,
    'Tipe Data': df.dtypes.values,
    'Non-Null Count': df.count().values,
    'Null Count': df.isnull().sum().values
})
display(info_df)

# Cek missing values
print(f"\n🔍 MISSING VALUES:")
print("=" * 60)
missing = df.isnull().sum()
if missing.sum() == 0:
    print("✅ Tidak ada missing values!")
else:
    print(missing[missing > 0])

# Statistik per tahun dalam bentuk tabel
print(f"\n📈 STATISTIK PER TAHUN:")
print("=" * 80)
yearly_stats = df.groupby('Tahun').agg({
    'Tanggal': 'count',
    kategori_beras[0]: 'mean',
    kategori_beras[1]: 'mean',
    kategori_beras[2]: 'mean',
    'Temperatur': 'mean',
    'Curah_Hujan': 'mean'
}).round(2)

yearly_stats.columns = ['Jumlah Data',
                        kategori_beras[0].replace('Beras Kualitas ', ''),
                        kategori_beras[1].replace('Beras Kualitas ', ''),
                        kategori_beras[2].replace('Beras Kualitas ', ''),
                        'Temperatur', 'Curah_Hujan']

display(yearly_stats)

# Sample data dengan format yang rapi
print(f"\n📋 SAMPLE DATA (Format Rapi):")
print("=" * 120)
sample_df = df.head(20).copy()
sample_df['Tanggal_Format'] = sample_df['Tanggal'].dt.strftime('%d/%m/%Y')

# Select columns untuk display
display_cols = ['Tanggal_Format', 'Tahun'] + kategori_beras + weather_features
sample_display = sample_df[display_cols]

# Rename untuk display yang lebih pendek
sample_display.columns = ['Tanggal', 'Tahun'] + [k.replace('Beras Kualitas ', '') for k in kategori_beras] + weather_features

display(sample_display)

# Save dataset info
dataset_info = {
    'total_rows': int(len(df)),
    'total_columns': int(len(df.columns)),
    'columns': list(df.columns),
    'kategori_beras': kategori_beras,
    'weather_features': weather_features,
    'periode': {
        'start': df['Tanggal'].min().strftime('%Y-%m-%d'),
        'end': df['Tanggal'].max().strftime('%Y-%m-%d'),
        'total_days': int(len(df))
    },
    'yearly_distribution': {
        str(year): {
            'count': int(stats['Jumlah Data']),
            'avg_price_sample': float(stats[yearly_stats.columns[1]])
        }
        for year, stats in yearly_stats.iterrows()
    },
    'missing_values': {k: int(v) for k, v in df.isnull().sum().to_dict().items()},
    'price_statistics': {
        kategori: {
            'min': float(df[kategori].min()),
            'max': float(df[kategori].max()),
            'mean': float(df[kategori].mean()),
            'median': float(df[kategori].median())
        }
        for kategori in kategori_beras
    },
    'weather_statistics': {
        feature: {
            'min': float(df[feature].min()),
            'max': float(df[feature].max()),
            'mean': float(df[feature].mean()),
            'median': float(df[feature].median())
        }
        for feature in weather_features
    },
    'weather_price_correlation': {
        kategori: {feature: float(weather_corr.loc[kategori, feature]) for feature in weather_features}
        for kategori in kategori_beras
    }
}

with open('flask_components/data/dataset_info.json', 'w', encoding='utf-8') as f:
    json.dump(dataset_info, f, indent=2, ensure_ascii=False)

# Save dataset
df.to_csv('flask_components/data/sorted_dataset.csv', index=False)

print("\n✅ Dataset (harga + cuaca) berhasil diload dan dieksplorasi!")
print("💾 Dataset info sudah disimpan!")
print(f"\n📊 Total: {len(df)} baris data dengan {len(kategori_beras)} kategori beras + {len(weather_features)} fitur cuaca")


📊 INFORMASI DATASET PREDIKSI HARGA BAHAN PANGAN KABUPATEN LANGKAT
Judul Penelitian: PENERAPAN METODE SUPPORT VECTOR REGRESSION
                  DALAM MEMPREDIKSI HARGA BAHAN PANGAN DI KABUPATEN LANGKAT
Jumlah baris: 1258
Jumlah kolom: 8

🌦️  MENGGABUNGKAN DATA CUACA HARIAN...
Data cuaca: 1827 hari (2020-01-01 s/d 2024-12-31)
Kolom cuaca: ['Tanggal', 'Temperatur', 'Curah_Hujan']
✅ Data cuaca berhasil digabungkan, tidak ada nilai kosong!

Periode data: 2020-01-02 s/d 2024-12-31
Total hari: 1258 hari
Rentang tahun: 2020 - 2024

📋 TABEL 4.2 DATASET PENELITIAN (Harga + Cuaca):

10 Data PERTAMA:


,Tahun,Tanggal,Beras Kualitas Bawah I,Beras Kualitas Bawah II,Beras Kualitas Medium I,Beras Kualitas Medium II,Beras Kualitas Super I,Beras Kualitas Super II,Temperatur,Curah_Hujan
0,2020,2020-01-02,9700,9950,11150,11200,12100,12000,25.508333,0.0
1,2020,2020-01-03,9700,9950,11150,11200,12100,12000,25.833333,1.5
2,2020,2020-01-06,9700,9950,11150,11200,12100,12000,25.645833,0.6
3,2020,2020-01-07,9700,9950,11150,11200,12100,12000,25.345833,1.5
4,2020,2020-01-08,9700,9950,11150,11150,12100,12000,25.595833,0.4
5,2020,2020-01-09,9700,9950,11150,11200,12100,12000,25.300000,12.8
6,2020,2020-01-10,9700,9950,11150,11200,12100,12000,25.904167,0.1
7,2020,2020-01-13,9700,9950,11150,11200,12100,12000,25.833333,0.2
8,2020,2020-01-14,9700,9950,11150,11200,12100,12000,26.129167,0.8
9,2020,2020-01-15,9700,9950,11150,11150,12100,12000,26.070833,0.3



10 Data TERAKHIR:


,Tahun,Tanggal,Beras Kualitas Bawah I,Beras Kualitas Bawah II,Beras Kualitas Medium I,Beras Kualitas Medium II,Beras Kualitas Super I,Beras Kualitas Super II,Temperatur,Curah_Hujan
1248,2024,2024-12-18,12350,12750,13850,14050,14350,14750,25.287500,15.8
1249,2024,2024-12-19,12350,12750,13850,14050,14350,14800,24.720833,10.1
1250,2024,2024-12-20,12350,12750,13850,14050,14350,14800,25.325000,17.1
1251,2024,2024-12-23,12350,12750,13850,14050,14350,14800,25.058333,24.3
1252,2024,2024-12-24,12350,12750,13850,14050,14350,14800,24.625000,31.8
1253,2024,2024-12-25,12800,13400,13700,14050,14000,13850,24.670833,15.8
1254,2024,2024-12-26,12800,13400,13700,14050,14000,14700,26.020833,0.0
1255,2024,2024-12-27,12350,12750,13850,14050,14400,14800,26.154167,0.2
1256,2024,2024-12-30,12350,12750,13850,14050,14400,14750,24.975000,12.7
1257,2024,2024-12-31,12350,12750,13850,14050,14400,14800,24.916667,22.6



📊 KATEGORI BERAS YANG DIPREDIKSI:
1. Beras Kualitas Bawah I
   Mean: Rp 10,618 | Min: Rp 9,500 | Max: Rp 13,000
2. Beras Kualitas Bawah II
   Mean: Rp 11,103 | Min: Rp 9,900 | Max: Rp 13,650
3. Beras Kualitas Medium I
   Mean: Rp 12,134 | Min: Rp 10,450 | Max: Rp 15,800
4. Beras Kualitas Medium II
   Mean: Rp 12,306 | Min: Rp 11,000 | Max: Rp 14,300
5. Beras Kualitas Super I
   Mean: Rp 12,988 | Min: Rp 11,150 | Max: Rp 14,800
6. Beras Kualitas Super II
   Mean: Rp 13,135 | Min: Rp 11,750 | Max: Rp 15,250

📊 STATISTIK DESKRIPTIF HARGA BERAS:


...


🔍 MISSING VALUES:
✅ Tidak ada missing values!

📈 STATISTIK PER TAHUN:


,Jumlah Data,Bawah I,Bawah II,Medium I,Temperatur,Curah_Hujan
Tahun,,,,,,
2020,244,9685.86,9956.97,11062.50,25.97,11.12
2021,248,9685.89,10117.54,11132.06,25.73,9.80
2022,246,9982.32,10442.07,11497.97,25.47,13.01
2023,258,11281.59,11976.55,12875.78,25.68,11.82
2024,262,12313.36,12864.50,13946.18,26.21,11.45



📋 SAMPLE DATA (Format Rapi):


,Tanggal,Tahun,Bawah I,Bawah II,Medium I,Medium II,Super I,Super II,Temperatur,Curah_Hujan
0,02/01/2020,2020,9700,9950,11150,11200,12100,12000,25.508333,0.0
1,03/01/2020,2020,9700,9950,11150,11200,12100,12000,25.833333,1.5
2,06/01/2020,2020,9700,9950,11150,11200,12100,12000,25.645833,0.6
3,07/01/2020,2020,9700,9950,11150,11200,12100,12000,25.345833,1.5
4,08/01/2020,2020,9700,9950,11150,11150,12100,12000,25.595833,0.4
5,09/01/2020,2020,9700,9950,11150,11200,12100,12000,25.300000,12.8
6,10/01/2020,2020,9700,9950,11150,11200,12100,12000,25.904167,0.1
7,13/01/2020,2020,9700,9950,11150,11200,12100,12000,25.833333,0.2
8,14/01/2020,2020,9700,9950,11150,11200,12100,12000,26.129167,0.8
9,15/01/2020,2020,9700,9950,11150,11150,12100,12000,26.070833,0.3



✅ Dataset (harga + cuaca) berhasil diload dan dieksplorasi!
💾 Dataset info sudah disimpan!

📊 Total: 1258 baris data dengan 6 kategori beras + 2 fitur cuaca


In [3]:
# Cell 3: Visualisasi Trend Harga Beras + Save

print("📊 Membuat visualisasi trend harga beras...")

# 1. Line chart trend semua kategori beras
fig_all_trends = go.Figure()

colors = ['#FF6B35', '#4ECDC4', '#45B7D1', '#F7DC6F', '#BB8FCE', '#85C1E2']

for i, kategori in enumerate(kategori_beras):
    fig_all_trends.add_trace(go.Scatter(
        x=df['Tanggal'],
        y=df[kategori],
        mode='lines',
        name=kategori,
        line=dict(width=2, color=colors[i % len(colors)]),
        hovertemplate='%{x|%Y-%m-%d}<br>Harga: Rp %{y:,.0f}<extra></extra>'
    ))

fig_all_trends.update_layout(
    title="Trend Harga Beras di Kabupaten Langkat (2020-2024)",
    xaxis_title="Tanggal",
    yaxis_title="Harga (Rp/kg)",
    font=dict(size=12),
    width=1200,
    height=600,
    yaxis={'tickformat': ',.0f'},
    hovermode='x unified',
    legend=dict(
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02
    )
)

fig_all_trends.show()

# 2. Perbandingan harga rata-rata per tahun untuk setiap kategori
yearly_avg = df.groupby('Tahun')[kategori_beras].mean().reset_index()

fig_yearly = go.Figure()

for i, kategori in enumerate(kategori_beras):
    fig_yearly.add_trace(go.Bar(
        name=kategori,
        x=yearly_avg['Tahun'],
        y=yearly_avg[kategori],
        marker_color=colors[i % len(colors)],
        hovertemplate='Tahun %{x}<br>Harga: Rp %{y:,.0f}<extra></extra>'
    ))

fig_yearly.update_layout(
    title="Perbandingan Rata-rata Harga Beras per Tahun",
    xaxis_title="Tahun",
    yaxis_title="Harga Rata-rata (Rp/kg)",
    barmode='group',
    font=dict(size=12),
    width=1000,
    height=600,
    yaxis={'tickformat': ',.0f'},
    xaxis={'dtick': 1}
)

fig_yearly.show()

# 3. Box plot distribusi harga per kategori
fig_box = go.Figure()

for i, kategori in enumerate(kategori_beras):
    fig_box.add_trace(go.Box(
        y=df[kategori],
        name=kategori,
        marker_color=colors[i % len(colors)],
        boxmean='sd'
    ))

fig_box.update_layout(
    title="Distribusi Harga Beras per Kategori",
    yaxis_title="Harga (Rp/kg)",
    font=dict(size=12),
    width=1000,
    height=600,
    yaxis={'tickformat': ',.0f'},
    showlegend=True
)

fig_box.show()

# 4. Heatmap korelasi antar kategori beras
correlation_matrix = df[kategori_beras].corr()

fig_corr = go.Figure(data=go.Heatmap(
    z=correlation_matrix.values,
    x=correlation_matrix.columns,
    y=correlation_matrix.columns,
    colorscale='RdBu',
    zmid=0,
    text=correlation_matrix.values.round(2),
    texttemplate='%{text}',
    textfont={"size": 10},
    colorbar=dict(title="Korelasi")
))

fig_corr.update_layout(
    title="Korelasi Harga Antar Kategori Beras",
    font=dict(size=10),
    width=800,
    height=700
)

fig_corr.show()

# Analisis trend per tahun
print("\n📈 ANALISIS TREND HARGA PER TAHUN:")
print("=" * 80)
for i in range(len(yearly_avg) - 1):
    current_year = int(yearly_avg.iloc[i]['Tahun'])
    next_year = int(yearly_avg.iloc[i + 1]['Tahun'])
    
    print(f"\n{current_year} → {next_year}:")
    for kategori in kategori_beras:
        current_price = yearly_avg.iloc[i][kategori]
        next_price = yearly_avg.iloc[i + 1][kategori]
        change = next_price - current_price
        change_pct = (change / current_price) * 100
        trend = "📈 NAIK" if change > 0 else "📉 TURUN" if change < 0 else "➡️ STABIL"
        
        print(f"  {kategori}: {trend}")
        print(f"    Perubahan: Rp {change:,.0f} ({change_pct:+.2f}%)")

# Save visualizations
fig_all_trends.write_html('flask_components/visualizations/trend_harga_semua_kategori.html')
fig_yearly.write_html('flask_components/visualizations/perbandingan_harga_per_tahun.html')
fig_box.write_html('flask_components/visualizations/distribusi_harga_per_kategori.html')
fig_corr.write_html('flask_components/visualizations/korelasi_harga_antar_kategori.html')

print("\n📊 Visualisasi trend harga berhasil ditampilkan!")
print("💾 Semua visualisasi sudah disimpan!")

# ===== VISUALISASI FAKTOR CUACA (TAMBAHAN SESUAI REVISI) =====
print("\n🌦️  Membuat visualisasi faktor cuaca terhadap harga beras...")

# 5. Trend Temperatur & Curah Hujan harian (dual axis)
fig_weather_trend = make_subplots(specs=[[{"secondary_y": True}]])

fig_weather_trend.add_trace(
    go.Scatter(
        x=df['Tanggal'], y=df['Temperatur'],
        mode='lines', name='Temperatur (°C)',
        line=dict(color='#E74C3C', width=1.5),
        hovertemplate='%{x|%Y-%m-%d}<br>Temperatur: %{y:.1f}°C<extra></extra>'
    ),
    secondary_y=False
)

fig_weather_trend.add_trace(
    go.Bar(
        x=df['Tanggal'], y=df['Curah_Hujan'],
        name='Curah Hujan (mm)',
        marker_color='#3498DB', opacity=0.4,
        hovertemplate='%{x|%Y-%m-%d}<br>Curah Hujan: %{y:.1f}mm<extra></extra>'
    ),
    secondary_y=True
)

fig_weather_trend.update_layout(
    title="Trend Temperatur dan Curah Hujan Harian (2020-2024)",
    font=dict(size=12),
    width=1200,
    height=600,
    hovermode='x unified',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig_weather_trend.update_yaxes(title_text="Temperatur (°C)", secondary_y=False)
fig_weather_trend.update_yaxes(title_text="Curah Hujan (mm)", secondary_y=True)

fig_weather_trend.show()

# 6. Korelasi Cuaca terhadap Harga Beras (bar chart, per kategori)
weather_corr_long = weather_corr.reset_index().melt(
    id_vars='index', var_name='Faktor Cuaca', value_name='Korelasi'
)
weather_corr_long.columns = ['Kategori', 'Faktor Cuaca', 'Korelasi']
weather_corr_long['Kategori'] = weather_corr_long['Kategori'].str.replace('Beras Kualitas ', '')

fig_weather_corr = px.bar(
    weather_corr_long,
    x='Kategori', y='Korelasi', color='Faktor Cuaca',
    barmode='group',
    title='Korelasi Faktor Cuaca terhadap Harga Beras per Kategori',
    color_discrete_sequence=['#E74C3C', '#3498DB']
)
fig_weather_corr.update_layout(
    width=1100, height=550, font=dict(size=12),
    yaxis_title='Koefisien Korelasi (Pearson)'
)
fig_weather_corr.add_hline(y=0, line_dash='dot', line_color='gray')

fig_weather_corr.show()

print(f"\n📌 CATATAN: Korelasi cuaca terhadap harga relatif lemah (|r| < 0.2),")
print(f"   menunjukkan cuaca bukan faktor dominan, namun tetap relevan")
print(f"   dimasukkan sebagai variabel eksogen sesuai arahan pembimbing.")

# Save weather visualizations
fig_weather_trend.write_html('flask_components/visualizations/trend_cuaca_harian.html')
fig_weather_corr.write_html('flask_components/visualizations/korelasi_cuaca_harga.html')

print("\n🌦️  Visualisasi faktor cuaca berhasil ditampilkan!")
print("💾 Visualisasi cuaca sudah disimpan!")

📊 Membuat visualisasi trend harga beras...



📈 ANALISIS TREND HARGA PER TAHUN:

2020 → 2021:
  Beras Kualitas Bawah I: 📈 NAIK
    Perubahan: Rp 0 (+0.00%)
  Beras Kualitas Bawah II: 📈 NAIK
    Perubahan: Rp 161 (+1.61%)
  Beras Kualitas Medium I: 📈 NAIK
    Perubahan: Rp 70 (+0.63%)
  Beras Kualitas Medium II: 📈 NAIK
    Perubahan: Rp 182 (+1.63%)
  Beras Kualitas Super I: 📈 NAIK
    Perubahan: Rp 157 (+1.31%)
  Beras Kualitas Super II: 📈 NAIK
    Perubahan: Rp 261 (+2.18%)

2021 → 2022:
  Beras Kualitas Bawah I: 📈 NAIK
    Perubahan: Rp 296 (+3.06%)
  Beras Kualitas Bawah II: 📈 NAIK
    Perubahan: Rp 325 (+3.21%)
  Beras Kualitas Medium I: 📈 NAIK
    Perubahan: Rp 366 (+3.29%)
  Beras Kualitas Medium II: 📈 NAIK
    Perubahan: Rp 324 (+2.86%)
  Beras Kualitas Super I: 📈 NAIK
    Perubahan: Rp 206 (+1.69%)
  Beras Kualitas Super II: 📈 NAIK
    Perubahan: Rp 172 (+1.40%)

2022 → 2023:
  Beras Kualitas Bawah I: 📈 NAIK
    Perubahan: Rp 1,299 (+13.02%)
  Beras Kualitas Bawah II: 📈 NAIK
    Perubahan: Rp 1,534 (+14.70%)
  Beras Kuali


📌 CATATAN: Korelasi cuaca terhadap harga relatif lemah (|r| < 0.2),
   menunjukkan cuaca bukan faktor dominan, namun tetap relevan
   dimasukkan sebagai variabel eksogen sesuai arahan pembimbing.



🌦️  Visualisasi faktor cuaca berhasil ditampilkan!
💾 Visualisasi cuaca sudah disimpan!


In [4]:
# Cell 4: Data Preprocessing dan Feature Engineering (Harga + Cuaca) + Save

print("🔄 Melakukan data preprocessing dan feature engineering...")

# Copy data untuk preprocessing
df_processed = df.copy()

# 1. Extract time-based features
df_processed['Year'] = df_processed['Tanggal'].dt.year
df_processed['Month'] = df_processed['Tanggal'].dt.month
df_processed['Day'] = df_processed['Tanggal'].dt.day
df_processed['DayOfWeek'] = df_processed['Tanggal'].dt.dayofweek  # 0=Monday, 6=Sunday
df_processed['DayOfYear'] = df_processed['Tanggal'].dt.dayofyear
df_processed['Quarter'] = df_processed['Tanggal'].dt.quarter

# 2. Create lag features (harga hari sebelumnya)
for kategori in kategori_beras:
    df_processed[f'{kategori}_lag1'] = df_processed[kategori].shift(1)
    df_processed[f'{kategori}_lag7'] = df_processed[kategori].shift(7)
    df_processed[f'{kategori}_lag30'] = df_processed[kategori].shift(30)

# 3. Create rolling mean features
for kategori in kategori_beras:
    df_processed[f'{kategori}_rolling_mean_7'] = df_processed[kategori].rolling(window=7).mean()
    df_processed[f'{kategori}_rolling_mean_30'] = df_processed[kategori].rolling(window=30).mean()

# 4. Fitur cuaca: nilai harian + rolling mean curah hujan
#    Curah hujan kumulatif (rolling) merepresentasikan dampak terhadap pasokan/panen,
#    yang efeknya ke harga tidak instan seperti suhu harian.
df_processed['Curah_Hujan_rolling_7'] = df_processed['Curah_Hujan'].rolling(window=7).mean()
df_processed['Curah_Hujan_rolling_30'] = df_processed['Curah_Hujan'].rolling(window=30).mean()
df_processed['Temperatur_rolling_7'] = df_processed['Temperatur'].rolling(window=7).mean()

weather_engineered_features = [
    'Temperatur', 'Curah_Hujan',
    'Temperatur_rolling_7', 'Curah_Hujan_rolling_7', 'Curah_Hujan_rolling_30'
]

# 5. Drop rows with NaN values (dari lag dan rolling)
df_processed = df_processed.dropna().reset_index(drop=True)

print(f"\n📊 HASIL PREPROCESSING:")
print("=" * 60)
print(f"Jumlah data setelah preprocessing: {len(df_processed)} records")
print(f"Data yang di-drop (karena lag/rolling): {len(df) - len(df_processed)} records")
print(f"Total features sekarang: {len(df_processed.columns)} kolom")

# Feature columns untuk model (tidak termasuk target)
base_features = ['Year', 'Month', 'Day', 'DayOfWeek', 'DayOfYear', 'Quarter']

# Untuk setiap kategori beras, kita akan membuat model terpisah
print(f"\n📋 FEATURE ENGINEERING SUMMARY:")
print("=" * 60)
print("Base Time Features:")
for i, feature in enumerate(base_features, 1):
    print(f"  {i}. {feature}")

print(f"\nWeather Features (faktor cuaca, sama untuk semua kategori):")
for i, feature in enumerate(weather_engineered_features, 1):
    print(f"  {i}. {feature}")

print(f"\nLag Features untuk setiap kategori:")
print(f"  - lag1 (1 hari sebelumnya)")
print(f"  - lag7 (7 hari sebelumnya)")
print(f"  - lag30 (30 hari sebelumnya)")

print(f"\nRolling Mean Features untuk setiap kategori:")
print(f"  - rolling_mean_7 (rata-rata 7 hari)")
print(f"  - rolling_mean_30 (rata-rata 30 hari)")

# Save preprocessing info
preprocessing_info = {
    'original_data_count': int(len(df)),
    'processed_data_count': int(len(df_processed)),
    'dropped_records': int(len(df) - len(df_processed)),
    'base_features': base_features,
    'weather_features': weather_engineered_features,
    'lag_features': ['lag1', 'lag7', 'lag30'],
    'rolling_features': ['rolling_mean_7', 'rolling_mean_30'],
    'total_features_created': int(len(df_processed.columns)),
    'preprocessing_steps': [
        'Extract time-based features (Year, Month, Day, DayOfWeek, DayOfYear, Quarter)',
        'Merge data cuaca harian (Temperatur, Curah_Hujan) berdasarkan Tanggal',
        'Create weather rolling features (Temperatur_rolling_7, Curah_Hujan_rolling_7, Curah_Hujan_rolling_30)',
        'Create lag features (1, 7, 30 hari) untuk setiap kategori beras',
        'Create rolling mean features (7, 30 hari) untuk setiap kategori beras',
        'Drop rows dengan NaN values dari lag dan rolling features'
    ]
}

with open('flask_components/data/preprocessing_info.json', 'w', encoding='utf-8') as f:
    json.dump(preprocessing_info, f, indent=2, ensure_ascii=False)

# Save processed dataset
df_processed.to_csv('flask_components/data/processed_dataset.csv', index=False)

print("\n✅ Data preprocessing dan feature engineering (termasuk fitur cuaca) selesai!")
print("💾 Preprocessing info dan processed data sudah disimpan!")


🔄 Melakukan data preprocessing dan feature engineering...

📊 HASIL PREPROCESSING:
Jumlah data setelah preprocessing: 1228 records
Data yang di-drop (karena lag/rolling): 30 records
Total features sekarang: 49 kolom

📋 FEATURE ENGINEERING SUMMARY:
Base Time Features:
  1. Year
  2. Month
  3. Day
  4. DayOfWeek
  5. DayOfYear
  6. Quarter

Weather Features (faktor cuaca, sama untuk semua kategori):
  1. Temperatur
  2. Curah_Hujan
  3. Temperatur_rolling_7
  4. Curah_Hujan_rolling_7
  5. Curah_Hujan_rolling_30

Lag Features untuk setiap kategori:
  - lag1 (1 hari sebelumnya)
  - lag7 (7 hari sebelumnya)
  - lag30 (30 hari sebelumnya)

Rolling Mean Features untuk setiap kategori:
  - rolling_mean_7 (rata-rata 7 hari)
  - rolling_mean_30 (rata-rata 30 hari)

✅ Data preprocessing dan feature engineering (termasuk fitur cuaca) selesai!
💾 Preprocessing info dan processed data sudah disimpan!


In [5]:
# Cell 5: Data Splitting (Harga + Cuaca) dan Tampilkan Detail Split + Save

print("🔄 Melakukan data splitting untuk setiap kategori beras...")

# Untuk SVR, kita akan membuat model untuk setiap kategori beras
# Split: 80% training, 20% testing
test_size = 0.2
random_state = 42

# Dictionary untuk menyimpan split data untuk setiap kategori
split_data = {}

print(f"\n📊 DATA SPLITTING CONFIGURATION:")
print("=" * 80)
print(f"Dataset: Time series harga beras ({len(df_processed)} records)")
print(f"Training data: {(1-test_size)*100:.0f}%")
print(f"Testing data: {test_size*100:.0f}%")
print(f"Metode: Chronological split (data terlama untuk training, terbaru untuk testing)")
print(f"\nCatatan: Untuk time series, kita menggunakan chronological split")
print(f"         bukan random split agar tidak ada data leakage")

# Tentukan split point berdasarkan chronological order
split_idx = int(len(df_processed) * (1 - test_size))

print(f"\n📅 SPLIT BERDASARKAN TANGGAL:")
print("=" * 80)
print(f"Total data: {len(df_processed)} hari")
print(f"Split index: {split_idx}")
print(f"Training period: {df_processed['Tanggal'].iloc[0].strftime('%Y-%m-%d')} s/d {df_processed['Tanggal'].iloc[split_idx-1].strftime('%Y-%m-%d')}")
print(f"Testing period:  {df_processed['Tanggal'].iloc[split_idx].strftime('%Y-%m-%d')} s/d {df_processed['Tanggal'].iloc[-1].strftime('%Y-%m-%d')}")

# Split untuk setiap kategori beras
for kategori in kategori_beras:
    # Feature columns untuk kategori ini (waktu + cuaca + lag/rolling harga)
    feature_cols = base_features + weather_engineered_features + [
        f'{kategori}_lag1',
        f'{kategori}_lag7',
        f'{kategori}_lag30',
        f'{kategori}_rolling_mean_7',
        f'{kategori}_rolling_mean_30'
    ]
    
    X = df_processed[feature_cols]
    y = df_processed[kategori]
    
    # Chronological split
    X_train = X.iloc[:split_idx]
    X_test = X.iloc[split_idx:]
    y_train = y.iloc[:split_idx]
    y_test = y.iloc[split_idx:]
    
    split_data[kategori] = {
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test,
        'feature_cols': feature_cols
    }
    
    print(f"\n📦 {kategori}:")
    print(f"  Training samples: {len(X_train)}")
    print(f"  Testing samples:  {len(X_test)}")
    print(f"  Features: {len(feature_cols)}")
    print(f"  Training mean: Rp {y_train.mean():,.0f}")
    print(f"  Testing mean:  Rp {y_test.mean():,.0f}")

# ==================== TAMPILKAN DATAFRAME TRAINING DATA ====================
print(f"\n" + "="*120)
print(f"📋 TABEL TRAINING DATA")
print("="*120)

# Buat DataFrame training untuk ditampilkan
train_display = df_processed.iloc[:split_idx].copy()
train_display['Tanggal_Format'] = train_display['Tanggal'].dt.strftime('%d/%m/%Y')

# Pilih kolom untuk display (Tanggal dan semua kategori beras)
train_cols = ['Tanggal_Format'] + kategori_beras
train_table = train_display[train_cols].copy()
train_table.columns = ['Tanggal'] + [k.replace('Beras Kualitas ', '') for k in kategori_beras]

# Reset index untuk tampilan yang rapi
train_table = train_table.reset_index(drop=True)

print(f"\n10 Data PERTAMA (Training Set):")
display(train_table.head(10))

print(f"\n10 Data TERAKHIR (Training Set):")
display(train_table.tail(10))

print(f"\n📊 Total Training Data: {len(train_table)} baris")

# ==================== TAMPILKAN DATAFRAME TESTING DATA ====================
print(f"\n" + "="*120)
print(f"📋 TABEL TESTING DATA")
print("="*120)

# Buat DataFrame testing untuk ditampilkan
test_display = df_processed.iloc[split_idx:].copy()
test_display['Tanggal_Format'] = test_display['Tanggal'].dt.strftime('%d/%m/%Y')

# Pilih kolom untuk display
test_cols = ['Tanggal_Format'] + kategori_beras
test_table = test_display[test_cols].copy()
test_table.columns = ['Tanggal'] + [k.replace('Beras Kualitas ', '') for k in kategori_beras]

# Reset index untuk tampilan yang rapi
test_table = test_table.reset_index(drop=True)

print(f"\n10 Data PERTAMA (Testing Set):")
display(test_table.head(10))

print(f"\n10 Data TERAKHIR (Testing Set):")
display(test_table.tail(10))

print(f"\n📊 Total Testing Data: {len(test_table)} baris")

# ==================== SUMMARY TABLE ====================
print(f"\n" + "="*120)
print(f"📊 SUMMARY SPLIT DATA")
print("="*120)

summary_split = pd.DataFrame({
    'Dataset': ['Training', 'Testing', 'Total'],
    'Jumlah Data': [len(train_table), len(test_table), len(df_processed)],
    'Persentase': [f"{(len(train_table)/len(df_processed)*100):.1f}%", 
                   f"{(len(test_table)/len(df_processed)*100):.1f}%", 
                   "100.0%"],
    'Periode Awal': [train_display['Tanggal'].min().strftime('%d/%m/%Y'),
                     test_display['Tanggal'].min().strftime('%d/%m/%Y'),
                     df_processed['Tanggal'].min().strftime('%d/%m/%Y')],
    'Periode Akhir': [train_display['Tanggal'].max().strftime('%d/%m/%Y'),
                      test_display['Tanggal'].max().strftime('%d/%m/%Y'),
                      df_processed['Tanggal'].max().strftime('%d/%m/%Y')]
})

display(summary_split)

# ==================== STATISTIK PER KATEGORI ====================
print(f"\n" + "="*120)
print(f"📊 STATISTIK HARGA PER KATEGORI (Training vs Testing)")
print("="*120)

stats_comparison = []
for kategori in kategori_beras:
    train_mean = split_data[kategori]['y_train'].mean()
    test_mean = split_data[kategori]['y_test'].mean()
    train_std = split_data[kategori]['y_train'].std()
    test_std = split_data[kategori]['y_test'].std()
    
    stats_comparison.append({
        'Kategori': kategori.replace('Beras Kualitas ', ''),
        'Train Mean': f"Rp {train_mean:,.0f}",
        'Test Mean': f"Rp {test_mean:,.0f}",
        'Train Std': f"Rp {train_std:,.0f}",
        'Test Std': f"Rp {test_std:,.0f}"
    })

stats_df = pd.DataFrame(stats_comparison)
display(stats_df)

# Save splitting info
splitting_info = {
    'test_size': float(test_size),
    'random_state': int(random_state),
    'splitting_method': 'Chronological split for time series (oldest data for training, newest for testing)',
    'total_records': int(len(df_processed)),
    'split_index': int(split_idx),
    'train_size': int(split_idx),
    'test_size': int(len(df_processed) - split_idx),
    'train_period': {
        'start': df_processed['Tanggal'].iloc[0].strftime('%Y-%m-%d'),
        'end': df_processed['Tanggal'].iloc[split_idx-1].strftime('%Y-%m-%d')
    },
    'test_period': {
        'start': df_processed['Tanggal'].iloc[split_idx].strftime('%Y-%m-%d'),
        'end': df_processed['Tanggal'].iloc[-1].strftime('%Y-%m-%d')
    },
    'categories': {
        kategori: {
            'train_samples': int(len(split_data[kategori]['X_train'])),
            'test_samples': int(len(split_data[kategori]['X_test'])),
            'features_count': int(len(split_data[kategori]['feature_cols'])),
            'train_mean': float(split_data[kategori]['y_train'].mean()),
            'test_mean': float(split_data[kategori]['y_test'].mean())
        }
        for kategori in kategori_beras
    }
}

with open('flask_components/data/splitting_info.json', 'w', encoding='utf-8') as f:
    json.dump(splitting_info, f, indent=2, ensure_ascii=False)

# Save split data
with open('flask_components/data/split_data.pkl', 'wb') as f:
    pickle.dump(split_data, f)

# Save training dan testing tables ke CSV
train_table.to_csv('flask_components/data/training_data_table.csv', index=False)
test_table.to_csv('flask_components/data/testing_data_table.csv', index=False)

print("\n✅ Data splitting berhasil!")
print("💾 Split data dan info sudah disimpan!")
print("💾 Training dan testing tables sudah disimpan ke CSV!")
print(f"\nCatatan: Time series split memastikan tidak ada data leakage")
print(f"         (model tidak pernah 'melihat' data masa depan saat training)")

🔄 Melakukan data splitting untuk setiap kategori beras...

📊 DATA SPLITTING CONFIGURATION:
Dataset: Time series harga beras (1228 records)
Training data: 80%
Testing data: 20%
Metode: Chronological split (data terlama untuk training, terbaru untuk testing)

Catatan: Untuk time series, kita menggunakan chronological split
         bukan random split agar tidak ada data leakage

📅 SPLIT BERDASARKAN TANGGAL:
Total data: 1228 hari
Split index: 982
Training period: 2020-02-13 s/d 2024-01-22
Testing period:  2024-01-23 s/d 2024-12-31

📦 Beras Kualitas Bawah I:
  Training samples: 982
  Testing samples:  246
  Features: 16
  Training mean: Rp 10,215
  Testing mean:  Rp 12,339

📦 Beras Kualitas Bawah II:
  Training samples: 982
  Testing samples:  246
  Features: 16
  Training mean: Rp 10,693
  Testing mean:  Rp 12,880

📦 Beras Kualitas Medium I:
  Training samples: 982
  Testing samples:  246
  Features: 16
  Training mean: Rp 11,706
  Testing mean:  Rp 13,964

📦 Beras Kualitas Medium II:
  T

,Tanggal,Bawah I,Bawah II,Medium I,Medium II,Super I,Super II
0,13/02/2020,9700,9950,11100,11200,12050,12000
1,14/02/2020,9700,9950,11100,11200,12050,12000
2,17/02/2020,9700,9950,11100,11200,12050,12000
3,18/02/2020,9700,9950,11100,11200,12050,12000
4,19/02/2020,9700,9950,11100,11200,12050,12000
5,20/02/2020,9700,9950,11100,11200,12050,12000
6,21/02/2020,9700,9950,10950,11200,12050,12000
7,24/02/2020,9700,9950,10950,11200,12050,12000
8,25/02/2020,9700,9950,10950,11200,12050,12000
9,26/02/2020,9700,9950,10950,11200,12050,12000



10 Data TERAKHIR (Training Set):


,Tanggal,Bawah I,Bawah II,Medium I,Medium II,Super I,Super II
972,09/01/2024,11950,12650,13700,13850,14300,14550
973,10/01/2024,11950,12650,13700,13850,14300,14550
974,11/01/2024,11950,12650,13700,13850,14300,14550
975,12/01/2024,11950,12650,13700,13850,14300,14550
976,15/01/2024,11950,12650,13700,13850,14300,14550
977,16/01/2024,11950,12650,13700,13850,14300,14550
978,17/01/2024,11950,12650,13700,13850,14300,14550
979,18/01/2024,11950,12650,13700,13850,14300,14550
980,19/01/2024,11950,12650,13700,13850,14300,14550
981,22/01/2024,11950,12650,13700,13850,14300,14550



📊 Total Training Data: 982 baris

📋 TABEL TESTING DATA

10 Data PERTAMA (Testing Set):


...


📊 Total Testing Data: 246 baris

📊 SUMMARY SPLIT DATA


,Dataset,Jumlah Data,Persentase,Periode Awal,Periode Akhir
0,Training,982,80.0%,13/02/2020,22/01/2024
1,Testing,246,20.0%,23/01/2024,31/12/2024
2,Total,1228,100.0%,13/02/2020,31/12/2024



📊 STATISTIK HARGA PER KATEGORI (Training vs Testing)


,Kategori,Train Mean,Test Mean,Train Std,Test Std
0,Bawah I,"Rp 10,215","Rp 12,339",Rp 737,Rp 150
1,Bawah II,"Rp 10,693","Rp 12,880",Rp 880,Rp 108
2,Medium I,"Rp 11,706","Rp 13,964",Rp 827,Rp 106
3,Medium II,"Rp 11,889","Rp 14,107",Rp 863,Rp 79
4,Super I,"Rp 12,619","Rp 14,574",Rp 694,Rp 146
5,Super II,"Rp 12,720","Rp 14,934",Rp 841,Rp 133



✅ Data splitting berhasil!
💾 Split data dan info sudah disimpan!
💾 Training dan testing tables sudah disimpan ke CSV!

Catatan: Time series split memastikan tidak ada data leakage
         (model tidak pernah 'melihat' data masa depan saat training)


In [6]:
# Cell 6: Training SVR Model untuk Setiap Kategori + Save

import time

print("🔄 Training Support Vector Regression (SVR) Model dengan kernel RBF...")
print("=" * 80)

# Dictionary untuk menyimpan model dan hasil
svr_models = {}
svr_scalers = {}
svr_results = {}
training_history = []  # Untuk menyimpan history training

# Function untuk menghitung MAPE
def calculate_mape(y_true, y_pred):
    """Menghitung Mean Absolute Percentage Error"""
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# Training SVR untuk setiap kategori beras
for i, kategori in enumerate(kategori_beras, 1):
    print(f"\n{'='*80}")
    print(f"🔧 TRAINING MODEL {i}/{len(kategori_beras)}: {kategori}")
    print(f"{'='*80}")
    
    # Ambil data split
    X_train = split_data[kategori]['X_train']
    X_test = split_data[kategori]['X_test']
    y_train = split_data[kategori]['y_train']
    y_test = split_data[kategori]['y_test']
    
    print(f"📦 Data Information:")
    print(f"  Training samples: {len(X_train)}")
    print(f"  Testing samples:  {len(X_test)}")
    print(f"  Features count:   {X_train.shape[1]}")
    print(f"  Training period:  {df_processed['Tanggal'].iloc[:split_idx].min().strftime('%Y-%m-%d')} to {df_processed['Tanggal'].iloc[split_idx-1].strftime('%Y-%m-%d')}")
    print(f"  Testing period:   {df_processed['Tanggal'].iloc[split_idx].strftime('%Y-%m-%d')} to {df_processed['Tanggal'].iloc[-1].strftime('%Y-%m-%d')}")
    
    # Feature scaling (PENTING untuk SVR!)
    print(f"\n🔄 Scaling features...")
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()
    
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled = scaler_X.transform(X_test)
    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
    
    print(f"✅ Feature scaling completed!")
    print(f"  X_train scaled shape: {X_train_scaled.shape}")
    print(f"  X_test scaled shape:  {X_test_scaled.shape}")
    
    # Inisialisasi SVR dengan kernel RBF
    svr_model = SVR(
        kernel='rbf',           # Radial Basis Function kernel
        C=100,                  # Regularization parameter
        epsilon=0.1,            # Epsilon in epsilon-SVR
        gamma='scale'           # Kernel coefficient
    )
    
    print(f"\n📊 Parameter Model SVR:")
    print(f"  - kernel: RBF (Radial Basis Function)")
    print(f"  - C (Regularization): {svr_model.C}")
    print(f"  - epsilon: {svr_model.epsilon}")
    print(f"  - gamma: {svr_model.gamma}")
    
    # Training model
    print(f"\n⏳ Training model...")
    start_time = time.time()
    svr_model.fit(X_train_scaled, y_train_scaled)
    training_time = time.time() - start_time
    
    print(f"✅ Training selesai!")
    print(f"⏱️  Waktu training: {training_time:.2f} detik")
    print(f"📊 Support vectors: {svr_model.n_support_.sum()} vectors")
    
    # Prediksi pada training set
    print(f"\n🔮 Melakukan prediksi pada training set...")
    y_train_pred_scaled = svr_model.predict(X_train_scaled)
    y_train_pred = scaler_y.inverse_transform(y_train_pred_scaled.reshape(-1, 1)).ravel()
    
    # Prediksi pada testing set
    print(f"🔮 Melakukan prediksi pada testing set...")
    start_time = time.time()
    y_test_pred_scaled = svr_model.predict(X_test_scaled)
    y_test_pred = scaler_y.inverse_transform(y_test_pred_scaled.reshape(-1, 1)).ravel()
    prediction_time = time.time() - start_time
    
    print(f"✅ Prediksi selesai!")
    print(f"⏱️  Waktu prediksi: {prediction_time:.4f} detik")
    
    # Hitung metrik evaluasi
    # Training metrics
    train_mae = mean_absolute_error(y_train, y_train_pred)
    train_mse = mean_squared_error(y_train, y_train_pred)
    train_rmse = np.sqrt(train_mse)
    train_mape = calculate_mape(y_train, y_train_pred)
    
    # Testing metrics
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    test_rmse = np.sqrt(test_mse)
    test_mape = calculate_mape(y_test, y_test_pred)
    
    print(f"\n📊 HASIL EVALUASI SVR - {kategori}")
    print("=" * 60)
    print("TRAINING SET:")
    print(f"  MAE:      Rp {train_mae:,.0f}")
    print(f"  RMSE:     Rp {train_rmse:,.0f}")
    print(f"  MAPE:     {train_mape:.2f}%")
    
    print("\nTESTING SET:")
    print(f"  MAE:      Rp {test_mae:,.0f}")
    print(f"  RMSE:     Rp {test_rmse:,.0f}")
    print(f"  MAPE:     {test_mape:.2f}%")
    
    # Interpretasi performa
    if test_mape <= 10:
        interpretasi = "Sangat Baik"
    elif test_mape <= 20:
        interpretasi = "Baik"
    elif test_mape <= 30:
        interpretasi = "Cukup"
    else:
        interpretasi = "Perlu Perbaikan"
    
    print(f"\n✨ Interpretasi: {interpretasi} (MAPE {test_mape:.2f}%)")
    
    # Simpan ke training history
    training_history.append({
        'No': i,
        'Kategori': kategori.replace('Beras Kualitas ', ''),
        'Train_Samples': len(X_train),
        'Test_Samples': len(X_test),
        'Features': X_train.shape[1],
        'Training_Time': f"{training_time:.2f}s",
        'Support_Vectors': int(svr_model.n_support_.sum()),
        'Train_MAE': train_mae,
        'Train_RMSE': train_rmse,
        'Train_MAPE': train_mape,
        'Test_MAE': test_mae,
        'Test_RMSE': test_rmse,
        'Test_MAPE': test_mape,
        'Interpretasi': interpretasi
    })
    
    # Simpan model dan hasil
    svr_models[kategori] = svr_model
    svr_scalers[kategori] = {
        'scaler_X': scaler_X,
        'scaler_y': scaler_y
    }
    
    svr_results[kategori] = {
        'model_params': {
            'kernel': 'rbf',
            'C': float(svr_model.C),
            'epsilon': float(svr_model.epsilon),
            'gamma': svr_model.gamma
        },
        'training_metrics': {
            'mae': float(train_mae),
            'mse': float(train_mse),
            'rmse': float(train_rmse),
            'mape': float(train_mape)
        },
        'testing_metrics': {
            'mae': float(test_mae),
            'mse': float(test_mse),
            'rmse': float(test_rmse),
            'mape': float(test_mape)
        },
        'timing': {
            'training_time': float(training_time),
            'prediction_time': float(prediction_time)
        },
        'predictions': {
            'y_train_pred': y_train_pred.tolist(),
            'y_test_pred': y_test_pred.tolist(),
            'y_train_actual': y_train.values.tolist(),
            'y_test_actual': y_test.values.tolist()
        },
        'support_vectors_count': int(svr_model.n_support_.sum()),
        'interpretasi': interpretasi
    }

# Save models dan scalers
with open('flask_components/models/svr_models.pkl', 'wb') as f:
    pickle.dump(svr_models, f)

with open('flask_components/models/svr_scalers.pkl', 'wb') as f:
    pickle.dump(svr_scalers, f)

# Save results
with open('flask_components/results/svr_results.json', 'w', encoding='utf-8') as f:
    json.dump(svr_results, f, indent=2, ensure_ascii=False)

print(f"\n{'='*80}")
print(f"✅ SEMUA MODEL SVR BERHASIL DITRAINING!")
print(f"{'='*80}")
print(f"Total model: {len(svr_models)}")
print(f"💾 Model SVR dan results sudah disimpan!")

# ==================== TABEL HISTORY TRAINING ====================
print(f"\n" + "="*120)
print(f"📋 TABEL HISTORY TRAINING - SEMUA KATEGORI")
print("="*120)

history_df = pd.DataFrame(training_history)
display(history_df)

# Save training history
history_df.to_csv('flask_components/results/training_history.csv', index=False)
print(f"\n💾 Training history saved to: flask_components/results/training_history.csv")

# ==================== TABEL METRIK EVALUASI (TRAINING SET) ====================
print(f"\n" + "="*120)
print(f"📊 TABEL METRIK EVALUASI - TRAINING SET")
print("="*120)

train_metrics = []
for kategori in kategori_beras:
    train_metrics.append({
        'Kategori': kategori.replace('Beras Kualitas ', ''),
        'MAE (Rp)': f"{svr_results[kategori]['training_metrics']['mae']:,.0f}",
        'RMSE (Rp)': f"{svr_results[kategori]['training_metrics']['rmse']:,.0f}",
        'MAPE (%)': f"{svr_results[kategori]['training_metrics']['mape']:.2f}%"
    })

train_metrics_df = pd.DataFrame(train_metrics)
display(train_metrics_df)

# Save training metrics table
train_metrics_df.to_csv('flask_components/results/training_metrics_table.csv', index=False)

# ==================== TABEL METRIK EVALUASI (TESTING SET) ====================
print(f"\n" + "="*120)
print(f"📊 TABEL METRIK EVALUASI - TESTING SET")
print("="*120)

test_metrics = []
for kategori in kategori_beras:
    test_metrics.append({
        'Kategori': kategori.replace('Beras Kualitas ', ''),
        'MAE (Rp)': f"{svr_results[kategori]['testing_metrics']['mae']:,.0f}",
        'RMSE (Rp)': f"{svr_results[kategori]['testing_metrics']['rmse']:,.0f}",
        'MAPE (%)': f"{svr_results[kategori]['testing_metrics']['mape']:.2f}%",
        'Interpretasi': svr_results[kategori]['interpretasi']
    })

test_metrics_df = pd.DataFrame(test_metrics)
display(test_metrics_df)

# Save testing metrics table
test_metrics_df.to_csv('flask_components/results/testing_metrics_table.csv', index=False)

# ==================== TABEL PERBANDINGAN TRAIN VS TEST ====================
print(f"\n" + "="*120)
print(f"📊 TABEL PERBANDINGAN METRIK: TRAINING vs TESTING")
print("="*120)

comparison_metrics = []
for kategori in kategori_beras:
    comparison_metrics.append({
        'Kategori': kategori.replace('Beras Kualitas ', ''),
        'Train MAE': f"{svr_results[kategori]['training_metrics']['mae']:,.0f}",
        'Test MAE': f"{svr_results[kategori]['testing_metrics']['mae']:,.0f}",
        'Train RMSE': f"{svr_results[kategori]['training_metrics']['rmse']:,.0f}",
        'Test RMSE': f"{svr_results[kategori]['testing_metrics']['rmse']:,.0f}",
        'Train MAPE': f"{svr_results[kategori]['training_metrics']['mape']:.2f}%",
        'Test MAPE': f"{svr_results[kategori]['testing_metrics']['mape']:.2f}%"
    })

comparison_df = pd.DataFrame(comparison_metrics)
display(comparison_df)

# Save comparison table
comparison_df.to_csv('flask_components/results/train_vs_test_comparison.csv', index=False)

# ==================== STATISTIK SUMMARY ====================
print(f"\n" + "="*120)
print(f"📊 STATISTIK SUMMARY")
print("="*120)

# Hitung rata-rata untuk semua kategori
avg_train_mape = np.mean([svr_results[k]['training_metrics']['mape'] for k in kategori_beras])
avg_test_mape = np.mean([svr_results[k]['testing_metrics']['mape'] for k in kategori_beras])
avg_train_mae = np.mean([svr_results[k]['training_metrics']['mae'] for k in kategori_beras])
avg_test_mae = np.mean([svr_results[k]['testing_metrics']['mae'] for k in kategori_beras])
avg_train_rmse = np.mean([svr_results[k]['training_metrics']['rmse'] for k in kategori_beras])
avg_test_rmse = np.mean([svr_results[k]['testing_metrics']['rmse'] for k in kategori_beras])

# Best dan worst category
best_category = min(kategori_beras, key=lambda k: svr_results[k]['testing_metrics']['mape'])
worst_category = max(kategori_beras, key=lambda k: svr_results[k]['testing_metrics']['mape'])

summary_stats = pd.DataFrame({
    'Metrik': ['Rata-rata MAE (Training)', 'Rata-rata MAE (Testing)',
               'Rata-rata RMSE (Training)', 'Rata-rata RMSE (Testing)',
               'Rata-rata MAPE (Training)', 'Rata-rata MAPE (Testing)',
               'Best Category (MAPE)', 'Worst Category (MAPE)'],
    'Nilai': [
        f"Rp {avg_train_mae:,.0f}",
        f"Rp {avg_test_mae:,.0f}",
        f"Rp {avg_train_rmse:,.0f}",
        f"Rp {avg_test_rmse:,.0f}",
        f"{avg_train_mape:.2f}%",
        f"{avg_test_mape:.2f}%",
        f"{best_category.replace('Beras Kualitas ', '')} ({svr_results[best_category]['testing_metrics']['mape']:.2f}%)",
        f"{worst_category.replace('Beras Kualitas ', '')} ({svr_results[worst_category]['testing_metrics']['mape']:.2f}%)"
    ]
})

display(summary_stats)

# Save summary stats
summary_stats.to_csv('flask_components/results/summary_statistics.csv', index=False)

print(f"\n✅ Semua tabel metrik evaluasi berhasil dibuat dan disimpan!")
print(f"💾 File CSV tersimpan di: flask_components/results/")

🔄 Training Support Vector Regression (SVR) Model dengan kernel RBF...

🔧 TRAINING MODEL 1/6: Beras Kualitas Bawah I
📦 Data Information:
  Training samples: 982
  Testing samples:  246
  Features count:   16
  Training period:  2020-02-13 to 2024-01-22
  Testing period:   2024-01-23 to 2024-12-31

🔄 Scaling features...
✅ Feature scaling completed!
  X_train scaled shape: (982, 16)
  X_test scaled shape:  (246, 16)

📊 Parameter Model SVR:
  - kernel: RBF (Radial Basis Function)
  - C (Regularization): 100
  - epsilon: 0.1
  - gamma: scale

⏳ Training model...
✅ Training selesai!
⏱️  Waktu training: 0.01 detik
📊 Support vectors: 199 vectors

🔮 Melakukan prediksi pada training set...
🔮 Melakukan prediksi pada testing set...
✅ Prediksi selesai!
⏱️  Waktu prediksi: 0.0016 detik

📊 HASIL EVALUASI SVR - Beras Kualitas Bawah I
TRAINING SET:
  MAE:      Rp 40
  RMSE:     Rp 47
  MAPE:     0.40%

TESTING SET:
  MAE:      Rp 867
  RMSE:     Rp 909
  MAPE:     7.01%

✨ Interpretasi: Sangat Baik (MA

✅ Feature scaling completed!
  X_train scaled shape: (982, 16)
  X_test scaled shape:  (246, 16)

📊 Parameter Model SVR:
  - kernel: RBF (Radial Basis Function)
  - C (Regularization): 100
  - epsilon: 0.1
  - gamma: scale

⏳ Training model...
✅ Training selesai!
⏱️  Waktu training: 0.01 detik
📊 Support vectors: 218 vectors

🔮 Melakukan prediksi pada training set...
🔮 Melakukan prediksi pada testing set...


✅ Prediksi selesai!
⏱️  Waktu prediksi: 0.0017 detik

📊 HASIL EVALUASI SVR - Beras Kualitas Super II
TRAINING SET:
  MAE:      Rp 47
  RMSE:     Rp 55
  MAPE:     0.37%

TESTING SET:
  MAE:      Rp 815
  RMSE:     Rp 867
  MAPE:     5.45%

✨ Interpretasi: Sangat Baik (MAPE 5.45%)



✅ SEMUA MODEL SVR BERHASIL DITRAINING!
Total model: 6
💾 Model SVR dan results sudah disimpan!

📋 TABEL HISTORY TRAINING - SEMUA KATEGORI


,No,Kategori,Train_Samples,Test_Samples,Features,Training_Time,Support_Vectors,Train_MAE,Train_RMSE,Train_MAPE,Test_MAE,Test_RMSE,Test_MAPE,Interpretasi
0,1,Bawah I,982,246,16,0.01s,199,40.495418,47.465717,0.395880,866.634922,909.061153,7.006805,Sangat Baik
1,2,Bawah II,982,246,16,0.01s,182,48.314136,56.244896,0.450910,690.896225,751.832289,5.355435,Sangat Baik
2,3,Medium I,982,246,16,0.01s,262,49.906928,56.952088,0.426753,725.485745,787.207841,5.187664,Sangat Baik
3,4,Medium II,982,246,16,0.01s,195,46.402347,54.570737,0.390784,715.923982,763.485870,5.070145,Sangat Baik
4,5,Super I,982,246,16,0.01s,236,40.365357,46.767494,0.321091,860.919069,925.113915,5.890265,Sangat Baik
5,6,Super II,982,246,16,0.01s,218,47.459679,55.410856,0.373532,815.385771,867.017856,5.450892,Sangat Baik


...


📊 TABEL PERBANDINGAN METRIK: TRAINING vs TESTING


,Kategori,Train MAE,Test MAE,Train RMSE,Test RMSE,Train MAPE,Test MAPE
0,Bawah I,40,867,47,909,0.40%,7.01%
1,Bawah II,48,691,56,752,0.45%,5.36%
2,Medium I,50,725,57,787,0.43%,5.19%
3,Medium II,46,716,55,763,0.39%,5.07%
4,Super I,40,861,47,925,0.32%,5.89%
5,Super II,47,815,55,867,0.37%,5.45%



📊 STATISTIK SUMMARY


,Metrik,Nilai
0,Rata-rata MAE (Training),Rp 45
1,Rata-rata MAE (Testing),Rp 779
2,Rata-rata RMSE (Training),Rp 53
3,Rata-rata RMSE (Testing),Rp 834
4,Rata-rata MAPE (Training),0.39%
5,Rata-rata MAPE (Testing),5.66%
6,Best Category (MAPE),Medium II (5.07%)
7,Worst Category (MAPE),Bawah I (7.01%)



✅ Semua tabel metrik evaluasi berhasil dibuat dan disimpan!
💾 File CSV tersimpan di: flask_components/results/


In [7]:
# Cell 7: Visualisasi Model Performance + Save

print("📊 Membuat visualisasi performance model SVR...")

# 1. Scatter plot Prediksi vs Aktual untuk setiap kategori
for kategori in kategori_beras:
    y_train_actual = np.array(svr_results[kategori]['predictions']['y_train_actual'])
    y_train_pred = np.array(svr_results[kategori]['predictions']['y_train_pred'])
    y_test_actual = np.array(svr_results[kategori]['predictions']['y_test_actual'])
    y_test_pred = np.array(svr_results[kategori]['predictions']['y_test_pred'])
    
    fig_scatter = go.Figure()
    
    # Testing data
    fig_scatter.add_trace(go.Scatter(
        x=y_test_actual,
        y=y_test_pred,
        mode='markers',
        name='Testing Data',
        marker=dict(color='#FF6B35', size=6, opacity=0.7),
        hovertemplate='Aktual: Rp %{x:,.0f}<br>Prediksi: Rp %{y:,.0f}<extra></extra>'
    ))
    
    # Training data
    fig_scatter.add_trace(go.Scatter(
        x=y_train_actual,
        y=y_train_pred,
        mode='markers',
        name='Training Data',
        marker=dict(color='#4ECDC4', size=4, opacity=0.5),
        hovertemplate='Aktual: Rp %{x:,.0f}<br>Prediksi: Rp %{y:,.0f}<extra></extra>'
    ))
    
    # Perfect prediction line
    min_val = min(min(y_test_actual), min(y_train_actual))
    max_val = max(max(y_test_actual), max(y_train_actual))
    
    fig_scatter.add_trace(go.Scatter(
        x=[min_val, max_val],
        y=[min_val, max_val],
        mode='lines',
        name='Perfect Prediction',
        line=dict(color='red', dash='dash', width=2)
    ))
    
    fig_scatter.update_layout(
        title=f"Prediksi vs Aktual - SVR Model ({kategori})",
        xaxis_title="Harga Aktual (Rp)",
        yaxis_title="Harga Prediksi (Rp)",
        width=800,
        height=600,
        font=dict(size=12),
        xaxis={'tickformat': ',.0f'},
        yaxis={'tickformat': ',.0f'}
    )
    
    fig_scatter.show()
    
    # Save visualization
    filename = kategori.replace(' ', '_').replace('/', '_')
    fig_scatter.write_html(f'flask_components/visualizations/prediksi_vs_aktual_{filename}.html')

# 2. Residual Plot untuk setiap kategori
for kategori in kategori_beras:
    y_test_actual = np.array(svr_results[kategori]['predictions']['y_test_actual'])
    y_test_pred = np.array(svr_results[kategori]['predictions']['y_test_pred'])
    residuals = y_test_actual - y_test_pred
    
    fig_residual = go.Figure()
    
    fig_residual.add_trace(go.Scatter(
        x=y_test_pred,
        y=residuals,
        mode='markers',
        marker=dict(color='#FF6B35', size=6, opacity=0.7),
        hovertemplate='Prediksi: Rp %{x:,.0f}<br>Residual: Rp %{y:,.0f}<extra></extra>'
    ))
    
    # Zero line
    fig_residual.add_hline(y=0, line_dash="dash", line_color="red")
    
    fig_residual.update_layout(
        title=f"Residual Plot - SVR Model ({kategori})",
        xaxis_title="Harga Prediksi (Rp)",
        yaxis_title="Residual (Aktual - Prediksi)",
        width=800,
        height=500,
        font=dict(size=12),
        xaxis={'tickformat': ',.0f'},
        yaxis={'tickformat': ',.0f'}
    )
    
    fig_residual.show()
    
    # Save visualization
    filename = kategori.replace(' ', '_').replace('/', '_')
    fig_residual.write_html(f'flask_components/visualizations/residual_plot_{filename}.html')

# 3. Bar chart perbandingan metrik untuk semua kategori
metrics_comparison = {
    'Kategori': [],
    'MAE': [],
    'RMSE': [],
    'MAPE': []
}

for kategori in kategori_beras:
    metrics_comparison['Kategori'].append(kategori.replace('Beras Kualitas ', ''))
    metrics_comparison['MAE'].append(svr_results[kategori]['testing_metrics']['mae'])
    metrics_comparison['RMSE'].append(svr_results[kategori]['testing_metrics']['rmse'])
    metrics_comparison['MAPE'].append(svr_results[kategori]['testing_metrics']['mape'])

# MAE Comparison
fig_mae = px.bar(
    x=metrics_comparison['Kategori'],
    y=metrics_comparison['MAE'],
    title="Perbandingan MAE untuk Semua Kategori Beras",
    labels={'x': 'Kategori Beras', 'y': 'MAE (Rp)'},
    color=metrics_comparison['MAE'],
    color_continuous_scale='Reds',
    text=[f"Rp {val:,.0f}" for val in metrics_comparison['MAE']]
)
fig_mae.update_traces(textposition='outside')
fig_mae.update_layout(width=1000, height=600, font=dict(size=12), showlegend=False)
fig_mae.show()
fig_mae.write_html('flask_components/visualizations/mae_comparison_all_categories.html')

# RMSE Comparison
fig_rmse = px.bar(
    x=metrics_comparison['Kategori'],
    y=metrics_comparison['RMSE'],
    title="Perbandingan RMSE untuk Semua Kategori Beras",
    labels={'x': 'Kategori Beras', 'y': 'RMSE (Rp)'},
    color=metrics_comparison['RMSE'],
    color_continuous_scale='Blues',
    text=[f"Rp {val:,.0f}" for val in metrics_comparison['RMSE']]
)
fig_rmse.update_traces(textposition='outside')
fig_rmse.update_layout(width=1000, height=600, font=dict(size=12), showlegend=False)
fig_rmse.show()
fig_rmse.write_html('flask_components/visualizations/rmse_comparison_all_categories.html')

# MAPE Comparison
fig_mape = px.bar(
    x=metrics_comparison['Kategori'],
    y=metrics_comparison['MAPE'],
    title="Perbandingan MAPE untuk Semua Kategori Beras",
    labels={'x': 'Kategori Beras', 'y': 'MAPE (%)'},
    color=metrics_comparison['MAPE'],
    color_continuous_scale='Greens',
    text=[f"{val:.2f}%" for val in metrics_comparison['MAPE']]
)
fig_mape.update_traces(textposition='outside')
fig_mape.update_layout(width=1000, height=600, font=dict(size=12), showlegend=False)
fig_mape.show()
fig_mape.write_html('flask_components/visualizations/mape_comparison_all_categories.html')

print("\n📊 Visualisasi model performance berhasil ditampilkan!")
print("💾 Semua visualisasi performance sudah disimpan!")

📊 Membuat visualisasi performance model SVR...



📊 Visualisasi model performance berhasil ditampilkan!
💾 Semua visualisasi performance sudah disimpan!


In [8]:
# Cell 8: Tabel Detail Hasil Prediksi vs Aktual

print("📋 MEMBUAT TABEL DETAIL HASIL PREDIKSI vs AKTUAL")
print("=" * 80)

# Ambil tanggal untuk testing period
test_dates = df_processed['Tanggal'].iloc[split_idx:].reset_index(drop=True)

# Ambil data cuaca untuk testing period (agar bisa ditampilkan berdampingan dengan harga)
test_weather = df_processed[['Temperatur', 'Curah_Hujan']].iloc[split_idx:].reset_index(drop=True)

# Untuk setiap kategori, buat tabel detail
for kategori in kategori_beras:
    print(f"\n{'='*120}")
    print(f"📊 TABEL DETAIL PREDIKSI vs AKTUAL - {kategori}")
    print(f"{'='*120}")
    
    y_test_actual = np.array(svr_results[kategori]['predictions']['y_test_actual'])
    y_test_pred = np.array(svr_results[kategori]['predictions']['y_test_pred'])
    
    # Buat DataFrame hasil prediksi
    results_df = pd.DataFrame({
        'Tanggal': test_dates,
        'Harga_Aktual': y_test_actual,
        'Harga_Prediksi': y_test_pred,
        'Error': y_test_pred - y_test_actual,
        'Absolute_Error': np.abs(y_test_pred - y_test_actual),
        'Percentage_Error': ((y_test_pred - y_test_actual) / y_test_actual) * 100,
        'Absolute_Percentage_Error': np.abs(((y_test_pred - y_test_actual) / y_test_actual) * 100),
        'Temperatur': test_weather['Temperatur'],
        'Curah_Hujan': test_weather['Curah_Hujan']
    })
    
    # ==================== TAMPILKAN FULL TABEL ====================
    print(f"\n📋 TABEL LENGKAP PREDIKSI vs AKTUAL ({len(results_df)} data):")
    print("="*120)
    
    # Format untuk display yang rapi
    display_df = results_df.copy()
    display_df['Tanggal'] = display_df['Tanggal'].dt.strftime('%d/%m/%Y')
    display_df['Harga_Aktual_Fmt'] = display_df['Harga_Aktual'].apply(lambda x: f"Rp {x:,.0f}")
    display_df['Harga_Prediksi_Fmt'] = display_df['Harga_Prediksi'].apply(lambda x: f"Rp {x:,.0f}")
    display_df['Error_Fmt'] = display_df['Error'].apply(lambda x: f"Rp {x:,.0f}")
    display_df['Absolute_Error_Fmt'] = display_df['Absolute_Error'].apply(lambda x: f"Rp {x:,.0f}")
    display_df['Percentage_Error_Fmt'] = display_df['Percentage_Error'].apply(lambda x: f"{x:.2f}%")
    display_df['APE_Fmt'] = display_df['Absolute_Percentage_Error'].apply(lambda x: f"{x:.2f}%")
    display_df['Temperatur_Fmt'] = display_df['Temperatur'].apply(lambda x: f"{x:.1f}\u00b0C")
    display_df['Curah_Hujan_Fmt'] = display_df['Curah_Hujan'].apply(lambda x: f"{x:.1f}mm")
    
    # Select kolom untuk display (termasuk cuaca, agar terlihat konteks Temperatur & Curah Hujan
    # di tanggal yang sama dengan harga aktual/prediksi)
    final_display = display_df[['Tanggal', 'Harga_Aktual_Fmt', 'Harga_Prediksi_Fmt', 
                                  'Error_Fmt', 'Absolute_Error_Fmt', 'Percentage_Error_Fmt', 'APE_Fmt',
                                  'Temperatur_Fmt', 'Curah_Hujan_Fmt']].copy()
    
    final_display.columns = ['Tanggal', 'Harga Aktual', 'Harga Prediksi', 
                             'Error', 'Absolute Error', 'Percentage Error (%)', 'APE (%)',
                             'Temperatur', 'Curah Hujan']
    
    # Tampilkan 20 data pertama
    print("\n20 Data PERTAMA:")
    display(final_display.head(20))
    
    # Tampilkan 20 data terakhir
    print(f"\n20 Data TERAKHIR:")
    display(final_display.tail(20))
    
    # Tampilkan middle section (jika ada)
    if len(final_display) > 50:
        mid_point = len(final_display) // 2
        print(f"\n20 Data TENGAH (sekitar data ke-{mid_point}):")
        display(final_display.iloc[mid_point-10:mid_point+10])
    
    # ==================== STATISTIK ERROR ====================
    print(f"\n📊 STATISTIK ERROR - {kategori}:")
    print("=" * 80)
    
    error_stats = pd.DataFrame({
        'Metrik': ['MAE', 'RMSE', 'MAPE', 'Min Error', 'Max Error', 'Mean Error', 'Std Error'],
        'Nilai': [
            f"Rp {results_df['Absolute_Error'].mean():,.0f}",
            f"Rp {np.sqrt((results_df['Error']**2).mean()):,.0f}",
            f"{results_df['Absolute_Percentage_Error'].mean():.2f}%",
            f"Rp {results_df['Error'].min():,.0f}",
            f"Rp {results_df['Error'].max():,.0f}",
            f"Rp {results_df['Error'].mean():,.0f}",
            f"Rp {results_df['Error'].std():,.0f}"
        ]
    })
    
    display(error_stats)
    
    # ==================== BEST PREDICTIONS ====================
    print(f"\n🏆 TOP 10 BEST PREDICTIONS (Error Terkecil):")
    print("=" * 120)
    
    best_pred = results_df.nsmallest(10, 'Absolute_Error').copy()
    best_pred['Tanggal'] = best_pred['Tanggal'].dt.strftime('%d/%m/%Y')
    best_pred_display = best_pred[['Tanggal', 'Harga_Aktual', 'Harga_Prediksi', 
                                     'Absolute_Error', 'Absolute_Percentage_Error']].copy()
    best_pred_display.columns = ['Tanggal', 'Aktual (Rp)', 'Prediksi (Rp)', 
                                   'Absolute Error (Rp)', 'APE (%)']
    best_pred_display['Aktual (Rp)'] = best_pred_display['Aktual (Rp)'].apply(lambda x: f"{x:,.0f}")
    best_pred_display['Prediksi (Rp)'] = best_pred_display['Prediksi (Rp)'].apply(lambda x: f"{x:,.0f}")
    best_pred_display['Absolute Error (Rp)'] = best_pred_display['Absolute Error (Rp)'].apply(lambda x: f"{x:,.0f}")
    best_pred_display['APE (%)'] = best_pred_display['APE (%)'].apply(lambda x: f"{x:.2f}%")
    
    display(best_pred_display.reset_index(drop=True))
    
    # ==================== WORST PREDICTIONS ====================
    print(f"\n❌ TOP 10 WORST PREDICTIONS (Error Terbesar):")
    print("=" * 120)
    
    worst_pred = results_df.nlargest(10, 'Absolute_Error').copy()
    worst_pred['Tanggal'] = worst_pred['Tanggal'].dt.strftime('%d/%m/%Y')
    worst_pred_display = worst_pred[['Tanggal', 'Harga_Aktual', 'Harga_Prediksi', 
                                       'Absolute_Error', 'Absolute_Percentage_Error']].copy()
    worst_pred_display.columns = ['Tanggal', 'Aktual (Rp)', 'Prediksi (Rp)', 
                                    'Absolute Error (Rp)', 'APE (%)']
    worst_pred_display['Aktual (Rp)'] = worst_pred_display['Aktual (Rp)'].apply(lambda x: f"{x:,.0f}")
    worst_pred_display['Prediksi (Rp)'] = worst_pred_display['Prediksi (Rp)'].apply(lambda x: f"{x:,.0f}")
    worst_pred_display['Absolute Error (Rp)'] = worst_pred_display['Absolute Error (Rp)'].apply(lambda x: f"{x:,.0f}")
    worst_pred_display['APE (%)'] = worst_pred_display['APE (%)'].apply(lambda x: f"{x:.2f}%")
    
    display(worst_pred_display.reset_index(drop=True))
    
    # ==================== DISTRIBUSI ERROR ====================
    print(f"\n📊 DISTRIBUSI ERROR:")
    print("=" * 80)
    
    # Klasifikasi berdasarkan APE
    error_ranges = [
        ('Sangat Baik (APE < 5%)', (results_df['Absolute_Percentage_Error'] < 5).sum()),
        ('Baik (5% ≤ APE < 10%)', ((results_df['Absolute_Percentage_Error'] >= 5) & 
                                     (results_df['Absolute_Percentage_Error'] < 10)).sum()),
        ('Cukup (10% ≤ APE < 20%)', ((results_df['Absolute_Percentage_Error'] >= 10) & 
                                       (results_df['Absolute_Percentage_Error'] < 20)).sum()),
        ('Buruk (APE ≥ 20%)', (results_df['Absolute_Percentage_Error'] >= 20).sum())
    ]
    
    error_dist = pd.DataFrame(error_ranges, columns=['Kategori Error', 'Jumlah Data'])
    error_dist['Persentase'] = (error_dist['Jumlah Data'] / len(results_df) * 100).apply(lambda x: f"{x:.2f}%")
    
    display(error_dist)
    
    # Save detailed results
    filename = kategori.replace(' ', '_').replace('/', '_')
    results_df.to_csv(f'flask_components/results/detailed_predictions_{filename}.csv', index=False)
    
    # Save formatted display table
    final_display.to_csv(f'flask_components/results/display_predictions_{filename}.csv', index=False)
    
    # Save best and worst predictions
    prediction_summary = {
        'kategori': kategori,
        'total_predictions': len(results_df),
        'statistics': {
            'mae': float(results_df['Absolute_Error'].mean()),
            'rmse': float(np.sqrt((results_df['Error']**2).mean())),
            'mape': float(results_df['Absolute_Percentage_Error'].mean()),
            'min_error': float(results_df['Error'].min()),
            'max_error': float(results_df['Error'].max()),
            'mean_error': float(results_df['Error'].mean()),
            'std_error': float(results_df['Error'].std())
        },
        'error_distribution': {
            'sangat_baik': int(error_ranges[0][1]),
            'baik': int(error_ranges[1][1]),
            'cukup': int(error_ranges[2][1]),
            'buruk': int(error_ranges[3][1])
        },
        'best_predictions': [
            {
                'date': row['Tanggal'].strftime('%Y-%m-%d'),
                'actual': float(row['Harga_Aktual']),
                'predicted': float(row['Harga_Prediksi']),
                'absolute_error': float(row['Absolute_Error']),
                'error_percentage': float(row['Absolute_Percentage_Error'])
            }
            for idx, row in results_df.nsmallest(10, 'Absolute_Error').iterrows()
        ],
        'worst_predictions': [
            {
                'date': row['Tanggal'].strftime('%Y-%m-%d'),
                'actual': float(row['Harga_Aktual']),
                'predicted': float(row['Harga_Prediksi']),
                'absolute_error': float(row['Absolute_Error']),
                'error_percentage': float(row['Absolute_Percentage_Error'])
            }
            for idx, row in results_df.nlargest(10, 'Absolute_Error').iterrows()
        ]
    }
    
    with open(f'flask_components/results/prediction_summary_{filename}.json', 'w', encoding='utf-8') as f:
        json.dump(prediction_summary, f, indent=2, ensure_ascii=False)
    
    print(f"\n💾 Tabel detail untuk {kategori} sudah disimpan!")

# ==================== RINGKASAN SEMUA KATEGORI ====================
print(f"\n" + "="*120)
print(f"📊 RINGKASAN PERFORMA PREDIKSI - SEMUA KATEGORI")
print("="*120)

summary_all = []
for kategori in kategori_beras:
    y_test_actual = np.array(svr_results[kategori]['predictions']['y_test_actual'])
    y_test_pred = np.array(svr_results[kategori]['predictions']['y_test_pred'])
    
    mae = mean_absolute_error(y_test_actual, y_test_pred)
    rmse = np.sqrt(mean_squared_error(y_test_actual, y_test_pred))
    mape = calculate_mape(y_test_actual, y_test_pred)
    
    summary_all.append({
        'Kategori': kategori.replace('Beras Kualitas ', ''),
        'Total Data': len(y_test_actual),
        'MAE (Rp)': f"{mae:,.0f}",
        'RMSE (Rp)': f"{rmse:,.0f}",
        'MAPE (%)': f"{mape:.2f}%"
    })

summary_all_df = pd.DataFrame(summary_all)
display(summary_all_df)

# Save summary
summary_all_df.to_csv('flask_components/results/prediction_summary_all_categories.csv', index=False)

print("\n✅ Tabel detail hasil prediksi vs aktual LENGKAP berhasil dibuat!")
print("💾 Semua detail predictions sudah disimpan!")
print(f"\n📁 File tersimpan di: flask_components/results/")
print(f"   - detailed_predictions_[kategori].csv (data mentah)")
print(f"   - display_predictions_[kategori].csv (data terformat)")
print(f"   - prediction_summary_[kategori].json (ringkasan + best/worst)")
print(f"   - prediction_summary_all_categories.csv (ringkasan semua kategori)")

📋 MEMBUAT TABEL DETAIL HASIL PREDIKSI vs AKTUAL

📊 TABEL DETAIL PREDIKSI vs AKTUAL - Beras Kualitas Bawah I

📋 TABEL LENGKAP PREDIKSI vs AKTUAL (246 data):

20 Data PERTAMA:


,Tanggal,Harga Aktual,Harga Prediksi,Error,Absolute Error,Percentage Error (%),APE (%),Temperatur,Curah Hujan
0,23/01/2024,"Rp 11,950","Rp 11,990",Rp 40,Rp 40,0.34%,0.34%,25.7°C,6.8mm
1,24/01/2024,"Rp 11,950","Rp 11,888",Rp -62,Rp 62,-0.52%,0.52%,25.5°C,23.2mm
2,25/01/2024,"Rp 11,950","Rp 11,743",Rp -207,Rp 207,-1.73%,1.73%,25.4°C,30.2mm
3,26/01/2024,"Rp 11,950","Rp 11,121",Rp -829,Rp 829,-6.94%,6.94%,25.1°C,56.7mm
4,29/01/2024,"Rp 11,950","Rp 11,753",Rp -197,Rp 197,-1.65%,1.65%,25.2°C,6.1mm
5,30/01/2024,"Rp 12,100","Rp 11,767",Rp -333,Rp 333,-2.76%,2.76%,26.0°C,1.2mm
6,31/01/2024,"Rp 12,100","Rp 11,635",Rp -465,Rp 465,-3.84%,3.84%,26.6°C,0.7mm
7,01/02/2024,"Rp 12,100","Rp 11,843",Rp -257,Rp 257,-2.13%,2.13%,26.1°C,4.4mm
8,02/02/2024,"Rp 12,100","Rp 11,814",Rp -286,Rp 286,-2.36%,2.36%,25.9°C,2.9mm
9,05/02/2024,"Rp 12,100","Rp 11,729",Rp -371,Rp 371,-3.07%,3.07%,27.4°C,0.2mm



20 Data TERAKHIR:


,Tanggal,Harga Aktual,Harga Prediksi,Error,Absolute Error,Percentage Error (%),APE (%),Temperatur,Curah Hujan
226,04/12/2024,"Rp 12,400","Rp 11,578",Rp -822,Rp 822,-6.63%,6.63%,25.0°C,14.3mm
227,05/12/2024,"Rp 12,400","Rp 11,522",Rp -878,Rp 878,-7.08%,7.08%,25.9°C,1.0mm
228,06/12/2024,"Rp 12,400","Rp 11,436",Rp -964,Rp 964,-7.78%,7.78%,25.9°C,4.7mm
229,09/12/2024,"Rp 12,400","Rp 11,531",Rp -869,Rp 869,-7.00%,7.00%,24.9°C,12.9mm
230,10/12/2024,"Rp 12,400","Rp 11,623",Rp -777,Rp 777,-6.27%,6.27%,24.8°C,25.7mm
231,11/12/2024,"Rp 12,400","Rp 11,689",Rp -711,Rp 711,-5.73%,5.73%,25.7°C,19.7mm
232,12/12/2024,"Rp 12,400","Rp 11,546",Rp -854,Rp 854,-6.89%,6.89%,26.3°C,2.3mm
233,13/12/2024,"Rp 12,400","Rp 11,623",Rp -777,Rp 777,-6.27%,6.27%,25.2°C,29.5mm
234,16/12/2024,"Rp 12,400","Rp 11,628",Rp -772,Rp 772,-6.23%,6.23%,25.8°C,11.3mm
235,17/12/2024,"Rp 12,350","Rp 11,712",Rp -638,Rp 638,-5.17%,5.17%,25.5°C,17.8mm



20 Data TENGAH (sekitar data ke-123):


...


📊 DISTRIBUSI ERROR:


,Kategori Error,Jumlah Data,Persentase
0,Sangat Baik (APE < 5%),107,43.50%
1,Baik (5% ≤ APE < 10%),132,53.66%
2,Cukup (10% ≤ APE < 20%),7,2.85%
3,Buruk (APE ≥ 20%),0,0.00%



💾 Tabel detail untuk Beras Kualitas Super II sudah disimpan!

📊 RINGKASAN PERFORMA PREDIKSI - SEMUA KATEGORI


,Kategori,Total Data,MAE (Rp),RMSE (Rp),MAPE (%)
0,Bawah I,246,867,909,7.01%
1,Bawah II,246,691,752,5.36%
2,Medium I,246,725,787,5.19%
3,Medium II,246,716,763,5.07%
4,Super I,246,861,925,5.89%
5,Super II,246,815,867,5.45%



✅ Tabel detail hasil prediksi vs aktual LENGKAP berhasil dibuat!
💾 Semua detail predictions sudah disimpan!

📁 File tersimpan di: flask_components/results/
   - detailed_predictions_[kategori].csv (data mentah)
   - display_predictions_[kategori].csv (data terformat)
   - prediction_summary_[kategori].json (ringkasan + best/worst)
   - prediction_summary_all_categories.csv (ringkasan semua kategori)


In [9]:
# Cell 9: Visualisasi Time Series Prediksi vs Aktual

print("📊 Membuat visualisasi time series prediksi vs aktual...")

# Ambil tanggal untuk training dan testing period
train_dates = df_processed['Tanggal'].iloc[:split_idx].reset_index(drop=True)
test_dates = df_processed['Tanggal'].iloc[split_idx:].reset_index(drop=True)

# Untuk setiap kategori, buat visualisasi time series
for kategori in kategori_beras:
    y_train_actual = np.array(svr_results[kategori]['predictions']['y_train_actual'])
    y_train_pred = np.array(svr_results[kategori]['predictions']['y_train_pred'])
    y_test_actual = np.array(svr_results[kategori]['predictions']['y_test_actual'])
    y_test_pred = np.array(svr_results[kategori]['predictions']['y_test_pred'])
    
    fig_ts = go.Figure()
    
    # Training data - Actual
    fig_ts.add_trace(go.Scatter(
        x=train_dates,
        y=y_train_actual,
        mode='lines',
        name='Training Actual',
        line=dict(color='#4ECDC4', width=2),
        hovertemplate='%{x|%Y-%m-%d}<br>Harga: Rp %{y:,.0f}<extra></extra>'
    ))
    
    # Training data - Predicted
    fig_ts.add_trace(go.Scatter(
        x=train_dates,
        y=y_train_pred,
        mode='lines',
        name='Training Predicted',
        line=dict(color='#4ECDC4', width=1, dash='dot'),
        hovertemplate='%{x|%Y-%m-%d}<br>Prediksi: Rp %{y:,.0f}<extra></extra>'
    ))
    
    # Testing data - Actual
    fig_ts.add_trace(go.Scatter(
        x=test_dates,
        y=y_test_actual,
        mode='lines',
        name='Testing Actual',
        line=dict(color='#FF6B35', width=3),
        hovertemplate='%{x|%Y-%m-%d}<br>Harga: Rp %{y:,.0f}<extra></extra>'
    ))
    
    # Testing data - Predicted
    fig_ts.add_trace(go.Scatter(
        x=test_dates,
        y=y_test_pred,
        mode='lines',
        name='Testing Predicted',
        line=dict(color='#45B7D1', width=2, dash='dash'),
        hovertemplate='%{x|%Y-%m-%d}<br>Prediksi: Rp %{y:,.0f}<extra></extra>'
    ))
    
    # Vertical line untuk split point - FIXED menggunakan shape
    split_date = df_processed['Tanggal'].iloc[split_idx]
    
    # Get y-axis range untuk vertical line
    all_values = np.concatenate([y_train_actual, y_train_pred, y_test_actual, y_test_pred])
    y_min = all_values.min()
    y_max = all_values.max()
    
    fig_ts.add_trace(go.Scatter(
        x=[split_date, split_date],
        y=[y_min, y_max],
        mode='lines',
        name='Train/Test Split',
        line=dict(color='gray', dash='dash', width=2),
        showlegend=True,
        hovertemplate='Split Point<extra></extra>'
    ))
    
    fig_ts.update_layout(
        title=f"Time Series: Prediksi vs Aktual - {kategori}",
        xaxis_title="Tanggal",
        yaxis_title="Harga (Rp)",
        width=1400,
        height=600,
        font=dict(size=12),
        yaxis={'tickformat': ',.0f'},
        hovermode='x unified',
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    
    fig_ts.show()
    
    # Save visualization
    filename = kategori.replace(' ', '_').replace('/', '_')
    fig_ts.write_html(f'flask_components/visualizations/time_series_prediksi_{filename}.html')

# Visualisasi semua kategori dalam satu plot (testing period only)
fig_all = go.Figure()

colors = ['#FF6B35', '#4ECDC4', '#45B7D1', '#F7DC6F', '#BB8FCE', '#85C1E2']

for i, kategori in enumerate(kategori_beras):
    y_test_actual = np.array(svr_results[kategori]['predictions']['y_test_actual'])
    y_test_pred = np.array(svr_results[kategori]['predictions']['y_test_pred'])
    
    # Actual
    fig_all.add_trace(go.Scatter(
        x=test_dates,
        y=y_test_actual,
        mode='lines',
        name=f'{kategori} (Actual)',
        line=dict(color=colors[i % len(colors)], width=2),
        hovertemplate='%{x|%Y-%m-%d}<br>Harga: Rp %{y:,.0f}<extra></extra>'
    ))
    
    # Predicted
    fig_all.add_trace(go.Scatter(
        x=test_dates,
        y=y_test_pred,
        mode='lines',
        name=f'{kategori} (Predicted)',
        line=dict(color=colors[i % len(colors)], width=1, dash='dash'),
        hovertemplate='%{x|%Y-%m-%d}<br>Prediksi: Rp %{y:,.0f}<extra></extra>'
    ))

fig_all.update_layout(
    title="Perbandingan Prediksi vs Aktual - Semua Kategori Beras (Testing Period)",
    xaxis_title="Tanggal",
    yaxis_title="Harga (Rp)",
    width=1400,
    height=700,
    font=dict(size=12),
    yaxis={'tickformat': ',.0f'},
    hovermode='x unified',
    legend=dict(
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02
    )
)

fig_all.show()
fig_all.write_html('flask_components/visualizations/time_series_all_categories_comparison.html')

print("\n📊 Visualisasi time series berhasil ditampilkan!")
print("💾 Semua visualisasi time series sudah disimpan!")

📊 Membuat visualisasi time series prediksi vs aktual...



📊 Visualisasi time series berhasil ditampilkan!
💾 Semua visualisasi time series sudah disimpan!


In [10]:
# Cell 10: Error Analysis per Tahun dan Kategori

print("🔍 ANALISIS ERROR BERDASARKAN TAHUN DAN KATEGORI")
print("=" * 80)

# Ambil tahun dari testing period
test_dates = df_processed['Tanggal'].iloc[split_idx:].reset_index(drop=True)
test_years = test_dates.dt.year

# Untuk setiap kategori, analisis error per tahun
yearly_error_analysis = {}

for kategori in kategori_beras:
    y_test_actual = np.array(svr_results[kategori]['predictions']['y_test_actual'])
    y_test_pred = np.array(svr_results[kategori]['predictions']['y_test_pred'])
    
    # Buat DataFrame untuk analisis
    error_df = pd.DataFrame({
        'Year': test_years,
        'Actual': y_test_actual,
        'Predicted': y_test_pred,
        'Error': y_test_pred - y_test_actual,
        'Absolute_Error': np.abs(y_test_pred - y_test_actual),
        'Absolute_Percentage_Error': np.abs(((y_test_pred - y_test_actual) / y_test_actual) * 100)
    })
    
    # Analisis per tahun
    yearly_stats = error_df.groupby('Year').agg({
        'Absolute_Error': ['mean', 'std', 'min', 'max'],
        'Absolute_Percentage_Error': 'mean'
    }).round(2)
    
    yearly_stats.columns = ['MAE', 'Std_Error', 'Min_Error', 'Max_Error', 'MAPE']
    yearly_stats = yearly_stats.reset_index()
    
    yearly_error_analysis[kategori] = yearly_stats
    
    print(f"\n📊 ERROR ANALYSIS - {kategori}")
    print("=" * 80)
    for _, row in yearly_stats.iterrows():
        print(f"Tahun {int(row['Year'])}:")
        print(f"  MAE:  Rp {row['MAE']:,.0f}")
        print(f"  MAPE: {row['MAPE']:.2f}%")
        print(f"  Error Range: Rp {row['Min_Error']:,.0f} - Rp {row['Max_Error']:,.0f}")

# Visualisasi Error per Tahun untuk semua kategori
all_yearly_errors = []
for kategori in kategori_beras:
    yearly_stats = yearly_error_analysis[kategori]
    for _, row in yearly_stats.iterrows():
        all_yearly_errors.append({
            'Kategori': kategori.replace('Beras Kualitas ', ''),
            'Tahun': int(row['Year']),
            'MAPE': row['MAPE'],
            'MAE': row['MAE']
        })

error_df_vis = pd.DataFrame(all_yearly_errors)

# MAPE per tahun
fig_mape_year = px.bar(
    error_df_vis,
    x='Tahun',
    y='MAPE',
    color='Kategori',
    barmode='group',
    title="MAPE per Tahun untuk Semua Kategori Beras",
    labels={'MAPE': 'MAPE (%)', 'Tahun': 'Tahun'},
    text='MAPE'
)
fig_mape_year.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig_mape_year.update_layout(width=1200, height=600, font=dict(size=12))
fig_mape_year.show()
fig_mape_year.write_html('flask_components/visualizations/mape_per_tahun_all_categories.html')

# MAE per tahun
fig_mae_year = px.line(
    error_df_vis,
    x='Tahun',
    y='MAE',
    color='Kategori',
    markers=True,
    title="MAE per Tahun untuk Semua Kategori Beras",
    labels={'MAE': 'MAE (Rp)', 'Tahun': 'Tahun'}
)
fig_mae_year.update_layout(width=1200, height=600, font=dict(size=12), yaxis={'tickformat': ',.0f'})
fig_mae_year.show()
fig_mae_year.write_html('flask_components/visualizations/mae_per_tahun_all_categories.html')

# Heatmap MAPE per kategori dan tahun
pivot_mape = error_df_vis.pivot(index='Kategori', columns='Tahun', values='MAPE')

fig_heatmap = px.imshow(
    pivot_mape,
    title="Heatmap MAPE per Kategori dan Tahun",
    labels={'x': 'Tahun', 'y': 'Kategori', 'color': 'MAPE (%)'},
    color_continuous_scale='RdYlBu_r',
    text_auto='.2f'
)
fig_heatmap.update_layout(width=900, height=600, font=dict(size=12))
fig_heatmap.show()
fig_heatmap.write_html('flask_components/visualizations/heatmap_mape_kategori_tahun.html')

# Save error analysis
error_analysis_data = {
    'yearly_error_by_category': {
        kategori: yearly_error_analysis[kategori].to_dict('records')
        for kategori in kategori_beras
    },
    'overall_insights': {
        'best_category': error_df_vis.groupby('Kategori')['MAPE'].mean().idxmin(),
        'worst_category': error_df_vis.groupby('Kategori')['MAPE'].mean().idxmax(),
        'average_mape_by_category': error_df_vis.groupby('Kategori')['MAPE'].mean().to_dict()
    }
}

with open('flask_components/results/error_analysis.json', 'w', encoding='utf-8') as f:
    json.dump(error_analysis_data, f, indent=2, ensure_ascii=False)

print("\n✅ Error analysis berhasil!")
print("💾 Error analysis dan visualizations sudah disimpan!")

🔍 ANALISIS ERROR BERDASARKAN TAHUN DAN KATEGORI

📊 ERROR ANALYSIS - Beras Kualitas Bawah I
Tahun 2024:
  MAE:  Rp 867
  MAPE: 7.01%
  Error Range: Rp 40 - Rp 1,723

📊 ERROR ANALYSIS - Beras Kualitas Bawah II
Tahun 2024:
  MAE:  Rp 691
  MAPE: 5.36%
  Error Range: Rp 3 - Rp 1,710

📊 ERROR ANALYSIS - Beras Kualitas Medium I
Tahun 2024:
  MAE:  Rp 725
  MAPE: 5.19%
  Error Range: Rp 9 - Rp 1,846

📊 ERROR ANALYSIS - Beras Kualitas Medium II
Tahun 2024:
  MAE:  Rp 716
  MAPE: 5.07%
  Error Range: Rp 6 - Rp 1,704

📊 ERROR ANALYSIS - Beras Kualitas Super I
Tahun 2024:
  MAE:  Rp 861
  MAPE: 5.89%
  Error Range: Rp 51 - Rp 1,884



📊 ERROR ANALYSIS - Beras Kualitas Super II


Tahun 2024:
  MAE:  Rp 815
  MAPE: 5.45%
  Error Range: Rp 17 - Rp 1,814



✅ Error analysis berhasil!
💾 Error analysis dan visualizations sudah disimpan!


In [11]:
# Cell 11: Model Performance Summary dan Comparison

print("📊 MODEL PERFORMANCE SUMMARY")
print("=" * 80)

# Summary metrics untuk semua kategori
performance_summary = {
    'research_info': {
        'title': 'PENERAPAN METODE SUPPORT VECTOR REGRESSION DALAM MEMPREDIKSI HARGA BAHAN PANGAN DI KABUPATEN LANGKAT',
        'method': 'Support Vector Regression (SVR) dengan kernel RBF',
        'categories': len(kategori_beras),
        'data_period': f"{df['Tanggal'].min().strftime('%Y-%m-%d')} s/d {df['Tanggal'].max().strftime('%Y-%m-%d')}",
        'total_records': len(df_processed),
        'train_size': split_idx,
        'test_size': len(df_processed) - split_idx
    },
    'model_parameters': {
        'kernel': 'RBF (Radial Basis Function)',
        'C': 100,
        'epsilon': 0.1,
        'gamma': 'scale'
    },
    'evaluation_metrics': ['MAE', 'RMSE', 'MAPE'],
    'results_by_category': {}
}

print("🎯 INFORMASI PENELITIAN:")
print("=" * 80)
print(f"Judul: {performance_summary['research_info']['title']}")
print(f"Metode: {performance_summary['research_info']['method']}")
print(f"Jumlah Kategori: {performance_summary['research_info']['categories']}")
print(f"Periode Data: {performance_summary['research_info']['data_period']}")
print(f"Total Records: {performance_summary['research_info']['total_records']}")

print(f"\n📈 HASIL EVALUASI FINAL (TESTING SET):")
print("=" * 80)
print(f"{'Kategori':<40} {'MAE':<15} {'RMSE':<15} {'MAPE':<10}")
print("-" * 80)

for kategori in kategori_beras:
    mae = svr_results[kategori]['testing_metrics']['mae']
    rmse = svr_results[kategori]['testing_metrics']['rmse']
    mape = svr_results[kategori]['testing_metrics']['mape']
    
    print(f"{kategori:<40} Rp {mae:>11,.0f} Rp {rmse:>11,.0f} {mape:>8.2f}%")
    
    performance_summary['results_by_category'][kategori] = {
        'mae': float(mae),
        'rmse': float(rmse),
        'mape': float(mape)
    }

# Rata-rata performa untuk semua kategori
avg_mae = np.mean([svr_results[k]['testing_metrics']['mae'] for k in kategori_beras])
avg_rmse = np.mean([svr_results[k]['testing_metrics']['rmse'] for k in kategori_beras])
avg_mape = np.mean([svr_results[k]['testing_metrics']['mape'] for k in kategori_beras])

print(f"\n{'RATA-RATA SEMUA KATEGORI:':<40} Rp {avg_mae:>11,.0f} Rp {avg_rmse:>11,.0f} {avg_mape:>8.2f}%")

performance_summary['average_performance'] = {
    'mae': float(avg_mae),
    'rmse': float(avg_rmse),
    'mape': float(avg_mape)
}

# Interpretasi hasil
print(f"\n🔍 INTERPRETASI HASIL:")
print("=" * 80)

if avg_mape <= 10:
    mape_interp = "Sangat Baik (≤10%)"
elif avg_mape <= 20:
    mape_interp = "Baik (10-20%)"
elif avg_mape <= 30:
    mape_interp = "Cukup (20-30%)"
else:
    mape_interp = "Perlu Perbaikan (>30%)"

print(f"MAPE Rata-rata: {mape_interp}")
print(f"\nModel SVR dapat memprediksi harga beras dengan:")
print(f"  - Error rata-rata: {avg_mape:.2f}%")
print(f"  - MAE: Rp {avg_mae:,.0f}")
print(f"  - RMSE: Rp {avg_rmse:,.0f}")

# Best dan Worst performing category
best_category = min(kategori_beras, key=lambda k: svr_results[k]['testing_metrics']['mape'])
worst_category = max(kategori_beras, key=lambda k: svr_results[k]['testing_metrics']['mape'])

print(f"\n🏆 Kategori dengan performa TERBAIK: {best_category}")
print(f"   MAPE: {svr_results[best_category]['testing_metrics']['mape']:.2f}%")

print(f"\n📉 Kategori dengan performa TERBURUK: {worst_category}")
print(f"   MAPE: {svr_results[worst_category]['testing_metrics']['mape']:.2f}%")

performance_summary['interpretation'] = {
    'mape_category': mape_interp,
    'best_category': {
        'name': best_category,
        'mape': float(svr_results[best_category]['testing_metrics']['mape'])
    },
    'worst_category': {
        'name': worst_category,
        'mape': float(svr_results[worst_category]['testing_metrics']['mape'])
    }
}

# Visualisasi Summary Metrics (Train vs Test)
comparison_data = {
    'Kategori': [],
    'Train_MAE': [],
    'Test_MAE': [],
    'Train_MAPE': [],
    'Test_MAPE': []
}

for kategori in kategori_beras:
    comparison_data['Kategori'].append(kategori.replace('Beras Kualitas ', ''))
    comparison_data['Train_MAE'].append(svr_results[kategori]['training_metrics']['mae'])
    comparison_data['Test_MAE'].append(svr_results[kategori]['testing_metrics']['mae'])
    comparison_data['Train_MAPE'].append(svr_results[kategori]['training_metrics']['mape'])
    comparison_data['Test_MAPE'].append(svr_results[kategori]['testing_metrics']['mape'])

# MAE comparison
fig_mae_comp = go.Figure()
fig_mae_comp.add_trace(go.Bar(
    name='Training',
    x=comparison_data['Kategori'],
    y=comparison_data['Train_MAE'],
    marker_color='#4ECDC4'
))
fig_mae_comp.add_trace(go.Bar(
    name='Testing',
    x=comparison_data['Kategori'],
    y=comparison_data['Test_MAE'],
    marker_color='#FF6B35'
))
fig_mae_comp.update_layout(
    title='Perbandingan MAE: Training vs Testing',
    xaxis_title='Kategori',
    yaxis_title='MAE (Rp)',
    barmode='group',
    width=1200,
    height=600,
    font=dict(size=12),
    yaxis={'tickformat': ',.0f'}
)
fig_mae_comp.show()
fig_mae_comp.write_html('flask_components/visualizations/mae_comparison_train_test.html')

# MAPE comparison
fig_mape_comp = go.Figure()
fig_mape_comp.add_trace(go.Bar(
    name='Training',
    x=comparison_data['Kategori'],
    y=comparison_data['Train_MAPE'],
    marker_color='#4ECDC4',
    text=[f'{v:.2f}%' for v in comparison_data['Train_MAPE']],
    textposition='outside'
))
fig_mape_comp.add_trace(go.Bar(
    name='Testing',
    x=comparison_data['Kategori'],
    y=comparison_data['Test_MAPE'],
    marker_color='#FF6B35',
    text=[f'{v:.2f}%' for v in comparison_data['Test_MAPE']],
    textposition='outside'
))
fig_mape_comp.update_layout(
    title='Perbandingan MAPE: Training vs Testing',
    xaxis_title='Kategori',
    yaxis_title='MAPE (%)',
    barmode='group',
    width=1200,
    height=600,
    font=dict(size=12)
)
fig_mape_comp.show()
fig_mape_comp.write_html('flask_components/visualizations/mape_comparison_train_test.html')

# Save performance summary
with open('flask_components/results/final_performance_summary.json', 'w', encoding='utf-8') as f:
    json.dump(performance_summary, f, indent=2, ensure_ascii=False)

print("\n✅ Model performance summary berhasil!")
print("💾 Performance summary sudah disimpan!")

📊 MODEL PERFORMANCE SUMMARY
🎯 INFORMASI PENELITIAN:
Judul: PENERAPAN METODE SUPPORT VECTOR REGRESSION DALAM MEMPREDIKSI HARGA BAHAN PANGAN DI KABUPATEN LANGKAT
Metode: Support Vector Regression (SVR) dengan kernel RBF
Jumlah Kategori: 6
Periode Data: 2020-01-02 s/d 2024-12-31
Total Records: 1228

📈 HASIL EVALUASI FINAL (TESTING SET):
Kategori                                 MAE             RMSE            MAPE      
--------------------------------------------------------------------------------
Beras Kualitas Bawah I                   Rp         867 Rp         909     7.01%
Beras Kualitas Bawah II                  Rp         691 Rp         752     5.36%
Beras Kualitas Medium I                  Rp         725 Rp         787     5.19%
Beras Kualitas Medium II                 Rp         716 Rp         763     5.07%
Beras Kualitas Super I                   Rp         861 Rp         925     5.89%
Beras Kualitas Super II                  Rp         815 Rp         867     5.45%

RATA-RATA SE


✅ Model performance summary berhasil!
💾 Performance summary sudah disimpan!


In [12]:
# Cell 12: Create Complete Pipeline untuk Flask + Save

print("🚀 MEMBUAT COMPLETE PIPELINE UNTUK FLASK APPLICATION")
print("=" * 80)

def create_prediction_pipeline():
    """Membuat pipeline lengkap untuk prediksi harga bahan pangan"""
    return {
        'models': svr_models,
        'scalers': svr_scalers,
        'split_data': split_data,
        'kategori_beras': kategori_beras,
        'base_features': base_features,
        'weather_features': weather_engineered_features,
        'results': svr_results
    }

def predict_price(input_data, kategori, pipeline):
    """
    Fungsi untuk memprediksi harga beras dari input data baru
    
    Args:
        input_data (dict): Dictionary berisi feature values
        kategori (str): Kategori beras yang akan diprediksi
        pipeline (dict): Trained pipeline components
    
    Returns:
        dict: Hasil prediksi dan confidence metrics
    """
    try:
        # Validasi kategori
        if kategori not in pipeline['kategori_beras']:
            return {
                'error': f'Kategori tidak valid. Pilih dari: {pipeline["kategori_beras"]}',
                'success': False
            }
        
        # Get model dan scaler untuk kategori ini
        model = pipeline['models'][kategori]
        scaler_X = pipeline['scalers'][kategori]['scaler_X']
        scaler_y = pipeline['scalers'][kategori]['scaler_y']
        
        # Get feature columns untuk kategori ini
        feature_cols = pipeline['split_data'][kategori]['feature_cols']
        
        # Create dataframe dari input
        input_df = pd.DataFrame([input_data])
        
        # Pastikan semua feature ada
        for col in feature_cols:
            if col not in input_df.columns:
                return {
                    'error': f'Feature "{col}" tidak ditemukan dalam input',
                    'success': False
                }
        
        # Select features sesuai training
        X_pred = input_df[feature_cols]
        
        # Scale features
        X_pred_scaled = scaler_X.transform(X_pred)
        
        # Prediksi
        y_pred_scaled = model.predict(X_pred_scaled)
        y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1))[0][0]
        
        # Confidence interval estimation (rough approximation using MAE)
        mae = pipeline['results'][kategori]['testing_metrics']['mae']
        confidence_lower = y_pred - mae
        confidence_upper = y_pred + mae
        
        return {
            'kategori': kategori,
            'prediction': float(y_pred),
            'confidence_interval': {
                'lower': float(max(0, confidence_lower)),
                'upper': float(confidence_upper)
            },
            'model_metrics': {
                'mae': float(mae),
                'rmse': float(pipeline['results'][kategori]['testing_metrics']['rmse']),
                'mape': float(pipeline['results'][kategori]['testing_metrics']['mape'])
            },
            'success': True
        }
        
    except Exception as e:
        return {
            'error': str(e),
            'success': False
        }

# Create dan save complete pipeline
complete_pipeline = create_prediction_pipeline()

with open('flask_components/models/complete_pipeline.pkl', 'wb') as f:
    pickle.dump(complete_pipeline, f)

# Save prediction functions
prediction_functions = {
    'predict_price': predict_price,
    'create_prediction_pipeline': create_prediction_pipeline
}

with open('flask_components/models/prediction_functions.pkl', 'wb') as f:
    pickle.dump(prediction_functions, f)

# Test prediction function dengan sample data
print("🧪 TESTING PREDICTION FUNCTION:")
print("=" * 60)

# Sample input untuk testing (dengan fitur cuaca + lag dan rolling features)
sample_input = {
    'Year': 2024,
    'Month': 12,
    'Day': 15,
    'DayOfWeek': 0,  # Monday
    'DayOfYear': 350,
    'Quarter': 4,
    'Temperatur': 26.0,
    'Curah_Hujan': 5.0,
    'Temperatur_rolling_7': 25.8,
    'Curah_Hujan_rolling_7': 4.5,
    'Curah_Hujan_rolling_30': 6.0,
    'Beras Kualitas Bawah I_lag1': 12350,
    'Beras Kualitas Bawah I_lag7': 12300,
    'Beras Kualitas Bawah I_lag30': 12200,
    'Beras Kualitas Bawah I_rolling_mean_7': 12325,
    'Beras Kualitas Bawah I_rolling_mean_30': 12250
}

# Test untuk kategori pertama
test_kategori = kategori_beras[0]
test_result = predict_price(sample_input, test_kategori, complete_pipeline)

if test_result['success']:
    pred_value = test_result['prediction']
    conf_lower = test_result['confidence_interval']['lower']
    conf_upper = test_result['confidence_interval']['upper']
    
    print(f"✅ Test berhasil!")
    print(f"Kategori: {test_kategori}")
    print(f"Input sample: Desember 2024")
    print(f"Prediksi: Rp {pred_value:,.0f}")
    print(f"Confidence Interval: Rp {conf_lower:,.0f} - Rp {conf_upper:,.0f}")
    print(f"Model MAPE: {test_result['model_metrics']['mape']:.2f}%")
else:
    print(f"❌ Test gagal: {test_result['error']}")

# Create Flask configuration
flask_config = {
    'model_paths': {
        'complete_pipeline': 'flask_components/models/complete_pipeline.pkl',
        'svr_models': 'flask_components/models/svr_models.pkl',
        'svr_scalers': 'flask_components/models/svr_scalers.pkl',
        'prediction_functions': 'flask_components/models/prediction_functions.pkl'
    },
    'data_paths': {
        'sorted_dataset': 'flask_components/data/sorted_dataset.csv',
        'processed_dataset': 'flask_components/data/processed_dataset.csv',
        'split_data': 'flask_components/data/split_data.pkl'
    },
    'results_paths': {
        'final_performance': 'flask_components/results/final_performance_summary.json',
        'error_analysis': 'flask_components/results/error_analysis.json',
        'svr_results': 'flask_components/results/svr_results.json'
    },
    'visualization_paths': {
        'trend_harga': 'flask_components/visualizations/trend_harga_semua_kategori.html',
        'mae_comparison': 'flask_components/visualizations/mae_comparison_all_categories.html',
        'mape_comparison': 'flask_components/visualizations/mape_comparison_all_categories.html',
        'time_series': 'flask_components/visualizations/time_series_all_categories_comparison.html'
    },
    'app_settings': {
        'title': 'Prediksi Harga Bahan Pangan Kabupaten Langkat',
        'description': 'Sistem prediksi harga bahan pangan menggunakan Support Vector Regression',
        'method': 'SVR dengan kernel RBF',
        'version': '1.0.0'
    }
}

with open('flask_components/flask_config.json', 'w', encoding='utf-8') as f:
    json.dump(flask_config, f, indent=2, ensure_ascii=False)

print("\n✅ Complete pipeline untuk Flask berhasil dibuat!")
print("💾 Pipeline dan Flask config sudah disimpan!")

🚀 MEMBUAT COMPLETE PIPELINE UNTUK FLASK APPLICATION
🧪 TESTING PREDICTION FUNCTION:
✅ Test berhasil!
Kategori: Beras Kualitas Bawah I
Input sample: Desember 2024
Prediksi: Rp 11,156
Confidence Interval: Rp 10,290 - Rp 12,023
Model MAPE: 7.01%

✅ Complete pipeline untuk Flask berhasil dibuat!
💾 Pipeline dan Flask config sudah disimpan!


In [13]:
# Cell 13: Requirements dan Setup Instructions

print("📦 MEMBUAT REQUIREMENTS DAN SETUP INSTRUCTIONS")
print("=" * 80)

# Create requirements.txt untuk Flask app
requirements = [
    'Flask==2.3.3',
    'pandas==1.5.3',
    'numpy==1.24.3',
    'scikit-learn==1.3.0',
    'plotly==5.15.0',
    'gunicorn==21.2.0'
]

with open('flask_components/requirements.txt', 'w') as f:
    for req in requirements:
        f.write(req + '\n')

print("📋 Requirements.txt berhasil dibuat!")

# Create setup instructions
setup_instructions = """
# SETUP INSTRUCTIONS - PREDIKSI HARGA BAHAN PANGAN KABUPATEN LANGKAT

## 1. INSTALASI DEPENDENCIES
```bash
pip install -r flask_components/requirements.txt
```

## 2. STRUKTUR PROJECT FLASK
```
your_flask_project/
├── app.py                          # Main Flask application
├── templates/                      # HTML templates
│   ├── index.html                 # Home page
│   ├── predict.html               # Prediction form
│   └── dashboard.html             # Analytics dashboard
├── static/                        # CSS, JS, images
│   ├── css/
│   └── js/
└── flask_components/              # ML components (copy dari Jupyter)
    ├── models/                    # Trained SVR models
    ├── data/                      # Datasets
    ├── results/                   # Analysis results
    └── visualizations/            # Interactive charts
```

## 3. SAMPLE FLASK APP CODE

### app.py
```python
from flask import Flask, render_template, request, jsonify
import pickle
import json
import pandas as pd

app = Flask(__name__)

# Load ML pipeline
with open('flask_components/models/complete_pipeline.pkl', 'rb') as f:
    pipeline = pickle.load(f)

with open('flask_components/models/prediction_functions.pkl', 'rb') as f:
    prediction_functions = pickle.load(f)

@app.route('/')
def home():
    return render_template('index.html')

@app.route('/predict', methods=['GET', 'POST'])
def predict():
    if request.method == 'POST':
        # Get form data
        kategori = request.form['kategori']
        
        input_data = {
            'Year': int(request.form['year']),
            'Month': int(request.form['month']),
            'Day': int(request.form['day']),
            'DayOfWeek': int(request.form['dayofweek']),
            'DayOfYear': int(request.form['dayofyear']),
            'Quarter': int(request.form['quarter']),
            f'{kategori}_lag1': float(request.form['lag1']),
            f'{kategori}_lag7': float(request.form['lag7']),
            f'{kategori}_lag30': float(request.form['lag30']),
            f'{kategori}_rolling_mean_7': float(request.form['rolling7']),
            f'{kategori}_rolling_mean_30': float(request.form['rolling30'])
        }
        
        # Predict
        result = prediction_functions['predict_price'](input_data, kategori, pipeline)
        
        return jsonify(result)
    
    # GET request - show form
    categories = pipeline['kategori_beras']
    return render_template('predict.html', categories=categories)

@app.route('/dashboard')
def dashboard():
    # Load analytics data
    with open('flask_components/results/final_performance_summary.json', 'r') as f:
        performance = json.load(f)
    
    return render_template('dashboard.html', performance=performance)

@app.route('/api/historical_data')
def historical_data():
    # Return historical data untuk charts
    df = pd.read_csv('flask_components/data/sorted_dataset.csv')
    
    # Convert ke format JSON
    data = {
        'dates': df['Tanggal'].tolist(),
        'prices': {
            kategori: df[kategori].tolist()
            for kategori in pipeline['kategori_beras']
        }
    }
    
    return jsonify(data)

if __name__ == '__main__':
    app.run(debug=True)
```

## 4. DEPLOYMENT

Untuk deployment ke production, gunakan:
```bash
gunicorn -w 4 -b 0.0.0.0:8000 app:app
```

## 5. FITUR YANG TERSEDIA

- ✅ Prediksi harga beras real-time untuk 6 kategori
- ✅ Support Vector Regression dengan kernel RBF
- ✅ Analytics dashboard dengan Plotly
- ✅ Time series visualization
- ✅ Model performance metrics (MAE, RMSE, MAPE)
- ✅ Error analysis per tahun
- ✅ RESTful API endpoints

## 6. CATATAN PENTING

### Feature Engineering
Model membutuhkan lag features dan rolling mean features:
- lag1: Harga 1 hari sebelumnya
- lag7: Harga 7 hari sebelumnya
- lag30: Harga 30 hari sebelumnya
- rolling_mean_7: Rata-rata 7 hari
- rolling_mean_30: Rata-rata 30 hari

### Kategori Beras
1. Beras Kualitas Bawah I
2. Beras Kualitas Bawah II
3. Beras Kualitas Medium I
4. Beras Kualitas Medium II
5. Beras Kualitas Super I
6. Beras Kualitas Super II

### Model Performance
- Metode: Support Vector Regression (SVR) dengan kernel RBF
- Metrik Evaluasi: MAE, RMSE, MAPE
- Data: Time series harian 2020-2024
"""

with open('flask_components/SETUP_INSTRUCTIONS.md', 'w', encoding='utf-8') as f:
    f.write(setup_instructions)

print("📖 Setup instructions berhasil dibuat!")

# Create file structure info
file_structure = {
    'total_files_created': 0,
    'folders': {
        'models': [],
        'data': [],
        'results': [],
        'visualizations': []
    }
}

# Count files in each folder
for folder in file_structure['folders'].keys():
    folder_path = f'flask_components/{folder}'
    if os.path.exists(folder_path):
        files = os.listdir(folder_path)
        file_structure['folders'][folder] = files
        file_structure['total_files_created'] += len(files)

with open('flask_components/file_structure_info.json', 'w', encoding='utf-8') as f:
    json.dump(file_structure, f, indent=2)

print(f"📊 Total {file_structure['total_files_created']} files berhasil dibuat untuk Flask!")
print("💾 Setup requirements dan instructions sudah disimpan!")

📦 MEMBUAT REQUIREMENTS DAN SETUP INSTRUCTIONS
📋 Requirements.txt berhasil dibuat!
📖 Setup instructions berhasil dibuat!
📊 Total 89 files berhasil dibuat untuk Flask!
💾 Setup requirements dan instructions sudah disimpan!


In [14]:
# Cell 14: Final Research Report dan Summary

print("📋 FINAL RESEARCH REPORT")
print("=" * 80)
print("Judul: PENERAPAN METODE SUPPORT VECTOR REGRESSION")
print("       DALAM MEMPREDIKSI HARGA BAHAN PANGAN DI KABUPATEN LANGKAT")
print("=" * 80)

# Create comprehensive final report
final_report = {
    'research_info': {
        'title': 'PENERAPAN METODE SUPPORT VECTOR REGRESSION DALAM MEMPREDIKSI HARGA BAHAN PANGAN DI KABUPATEN LANGKAT',
        'researcher': 'Haris',
        'location': 'Kabupaten Langkat',
        'method': 'Support Vector Regression (SVR) dengan kernel RBF',
        'period': f"{df['Tanggal'].min().strftime('%Y-%m-%d')} s/d {df['Tanggal'].max().strftime('%Y-%m-%d')}",
        'completion_date': pd.Timestamp.now().isoformat()
    },
    'dataset_summary': {
        'total_records_original': int(len(df)),
        'total_records_processed': int(len(df_processed)),
        'categories_count': len(kategori_beras),
        'categories_list': kategori_beras,
        'time_period': {
            'start': df['Tanggal'].min().strftime('%Y-%m-%d'),
            'end': df['Tanggal'].max().strftime('%Y-%m-%d'),
            'total_days': int(len(df))
        },
        'features_engineered': {
            'base_features': base_features,
            'weather_features': weather_engineered_features,
            'lag_features': ['lag1', 'lag7', 'lag30'],
            'rolling_features': ['rolling_mean_7', 'rolling_mean_30']
        }
    },
    'methodology': {
        'algorithm': 'Support Vector Regression (SVR)',
        'kernel': 'RBF (Radial Basis Function)',
        'parameters': {
            'C': 100,
            'epsilon': 0.1,
            'gamma': 'scale'
        },
        'data_split': 'Chronological split (80% Training, 20% Testing)',
        'evaluation_metrics': ['MAE', 'RMSE', 'MAPE'],
        'preprocessing': [
            'Time-based feature extraction',
            'Merge data cuaca harian (Temperatur, Curah Hujan)',
            'Weather rolling features (7, 30 hari)',
            'Lag features creation (1, 7, 30 hari)',
            'Rolling mean features (7, 30 hari)',
            'StandardScaler untuk normalisasi'
        ]
    },
    'results': {
        'average_performance': {
            'mae': float(avg_mae),
            'rmse': float(avg_rmse),
            'mape': float(avg_mape)
        },
        'best_category': {
            'name': best_category,
            'mape': float(svr_results[best_category]['testing_metrics']['mape']),
            'mae': float(svr_results[best_category]['testing_metrics']['mae']),
            'rmse': float(svr_results[best_category]['testing_metrics']['rmse'])
        },
        'worst_category': {
            'name': worst_category,
            'mape': float(svr_results[worst_category]['testing_metrics']['mape']),
            'mae': float(svr_results[worst_category]['testing_metrics']['mae']),
            'rmse': float(svr_results[worst_category]['testing_metrics']['rmse'])
        },
        'all_categories_performance': {
            kategori: {
                'mae': float(svr_results[kategori]['testing_metrics']['mae']),
                'rmse': float(svr_results[kategori]['testing_metrics']['rmse']),
                'mape': float(svr_results[kategori]['testing_metrics']['mape'])
            }
            for kategori in kategori_beras
        }
    },
    'conclusions': {
        'model_accuracy': f"Model dapat memprediksi harga beras dengan error rata-rata {avg_mape:.2f}%",
        'interpretasi': mape_interp,
        'practical_application': "Model SVR siap digunakan untuk prediksi harga bahan pangan di Kabupaten Langkat",
        'recommendations': [
            f"Model menunjukkan performa {mape_interp.lower()} untuk prediksi harga beras",
            f"Kategori terbaik: {best_category} dengan MAPE {svr_results[best_category]['testing_metrics']['mape']:.2f}%",
            "Sistem dapat diintegrasikan dengan web application untuk prediksi real-time",
            "Perlu update data berkala untuk maintenance akurasi model",
            "SVR dengan kernel RBF efektif untuk data time series harga bahan pangan"
        ]
    },
    'technical_deliverables': {
        'flask_application': 'Complete web application dengan prediction API',
        'ml_models': f'{len(kategori_beras)} SVR models (satu untuk setiap kategori)',
        'analytics_dashboard': 'Interactive dashboard dengan Plotly visualizations',
        'api_endpoints': 'RESTful API untuk prediksi dan historical data'
    }
}

print("📊 RINGKASAN HASIL PENELITIAN:")
print("=" * 60)
print(f"Dataset: {final_report['dataset_summary']['total_records_original']} hari data")
print(f"Periode: {final_report['research_info']['period']}")
print(f"Kategori: {final_report['dataset_summary']['categories_count']} kategori beras")
print(f"Metode: {final_report['research_info']['method']}")

print(f"\n🎯 PERFORMA MODEL (Rata-rata):")
print("=" * 60)
print(f"MAE:  Rp {avg_mae:,.0f}")
print(f"RMSE: Rp {avg_rmse:,.0f}")
print(f"MAPE: {avg_mape:.2f}%")

print(f"\n🏆 BEST PERFORMING CATEGORY:")
print("=" * 60)
print(f"{best_category}")
print(f"  MAPE: {svr_results[best_category]['testing_metrics']['mape']:.2f}%")
print(f"  MAE:  Rp {svr_results[best_category]['testing_metrics']['mae']:,.0f}")

print(f"\n📉 WORST PERFORMING CATEGORY:")
print("=" * 60)
print(f"{worst_category}")
print(f"  MAPE: {svr_results[worst_category]['testing_metrics']['mape']:.2f}%")
print(f"  MAE:  Rp {svr_results[worst_category]['testing_metrics']['mae']:,.0f}")

print(f"\n📈 DETAIL PERFORMA PER KATEGORI:")
print("=" * 80)
print(f"{'Kategori':<40} {'MAPE':<12} {'MAE':<15} {'RMSE':<15}")
print("-" * 80)
for kategori in kategori_beras:
    mape = svr_results[kategori]['testing_metrics']['mape']
    mae = svr_results[kategori]['testing_metrics']['mae']
    rmse = svr_results[kategori]['testing_metrics']['rmse']
    print(f"{kategori:<40} {mape:>10.2f}% Rp {mae:>11,.0f} Rp {rmse:>11,.0f}")

print(f"\n🚀 DELIVERABLES:")
print("=" * 60)
for key, value in final_report['technical_deliverables'].items():
    print(f"✅ {key.replace('_', ' ').title()}: {value}")

# Save final research report
with open('flask_components/FINAL_RESEARCH_REPORT.json', 'w', encoding='utf-8') as f:
    json.dump(final_report, f, indent=2, ensure_ascii=False)

# Save summary untuk publikasi
research_summary = f"""
# PENERAPAN METODE SUPPORT VECTOR REGRESSION
## DALAM MEMPREDIKSI HARGA BAHAN PANGAN DI KABUPATEN LANGKAT

**Peneliti:** Haris  
**Periode Data:** {final_report['research_info']['period']}  
**Completed:** {pd.Timestamp.now().strftime('%Y-%m-%d')}

### ABSTRACT

Penelitian ini mengembangkan model prediksi harga bahan pangan (beras) di Kabupaten Langkat menggunakan algoritma Support Vector Regression (SVR) dengan kernel Radial Basis Function (RBF). Dataset terdiri dari {final_report['dataset_summary']['total_records_original']} hari data time series dengan {final_report['dataset_summary']['categories_count']} kategori beras berbeda.

### METODOLOGI

- **Algoritma:** Support Vector Regression (SVR)
- **Kernel:** RBF (Radial Basis Function)
- **Parameter:** C=100, epsilon=0.1, gamma='scale'
- **Features:** Time-based features, lag features (1, 7, 30 hari), rolling mean (7, 30 hari)
- **Data Split:** Chronological split - 80% training, 20% testing
- **Preprocessing:** StandardScaler untuk normalisasi features

### HASIL UTAMA

**Performa Rata-rata:**
- **MAE:** Rp {avg_mae:,.0f}
- **RMSE:** Rp {avg_rmse:,.0f}
- **MAPE:** {avg_mape:.2f}%

**Kategori Terbaik:** {best_category}
- MAPE: {svr_results[best_category]['testing_metrics']['mape']:.2f}%

**Kategori Terburuk:** {worst_category}
- MAPE: {svr_results[worst_category]['testing_metrics']['mape']:.2f}%

### KESIMPULAN

Model SVR menunjukkan performa {mape_interp.lower()} untuk prediksi harga beras dengan tingkat error rata-rata {avg_mape:.2f}%. Sistem telah diimplementasikan dalam bentuk web application dengan Flask untuk prediksi real-time.

### IMPLEMENTASI

- ✅ {len(kategori_beras)} SVR Models (satu untuk setiap kategori)
- ✅ Web Application dengan Flask
- ✅ Real-time Prediction API
- ✅ Analytics Dashboard dengan Plotly
- ✅ Time Series Visualization

**Keywords:** Support Vector Regression, SVR, RBF Kernel, Price Prediction, Food Commodities, Time Series, Machine Learning

**Metrik Evaluasi:** MAE (Mean Absolute Error), RMSE (Root Mean Square Error), MAPE (Mean Absolute Percentage Error)
"""

with open('flask_components/RESEARCH_SUMMARY.md', 'w', encoding='utf-8') as f:
    f.write(research_summary)

print("\n✅ Final research report berhasil dibuat!")
print("💾 Complete research documentation sudah disimpan!")

📋 FINAL RESEARCH REPORT
Judul: PENERAPAN METODE SUPPORT VECTOR REGRESSION
       DALAM MEMPREDIKSI HARGA BAHAN PANGAN DI KABUPATEN LANGKAT
📊 RINGKASAN HASIL PENELITIAN:
Dataset: 1258 hari data
Periode: 2020-01-02 s/d 2024-12-31
Kategori: 6 kategori beras
Metode: Support Vector Regression (SVR) dengan kernel RBF

🎯 PERFORMA MODEL (Rata-rata):
MAE:  Rp 779
RMSE: Rp 834
MAPE: 5.66%

🏆 BEST PERFORMING CATEGORY:
Beras Kualitas Medium II
  MAPE: 5.07%
  MAE:  Rp 716

📉 WORST PERFORMING CATEGORY:
Beras Kualitas Bawah I
  MAPE: 7.01%
  MAE:  Rp 867

📈 DETAIL PERFORMA PER KATEGORI:
Kategori                                 MAPE         MAE             RMSE           
--------------------------------------------------------------------------------
Beras Kualitas Bawah I                         7.01% Rp         867 Rp         909
Beras Kualitas Bawah II                        5.36% Rp         691 Rp         752
Beras Kualitas Medium I                        5.19% Rp         725 Rp         787
Bera

In [15]:
# Cell 15: Verification dan Final Checklist

print("\n" + "="*80)
print("🧪 VERIFICATION - TESTING ALL COMPONENTS")
print("="*80)

verification_results = {}

# Test 1: Load model components
try:
    with open('flask_components/models/complete_pipeline.pkl', 'rb') as f:
        test_pipeline = pickle.load(f)
    verification_results['pipeline_loading'] = '✅ PASS'
    print("✅ Complete pipeline loaded successfully")
except Exception as e:
    verification_results['pipeline_loading'] = f'❌ FAIL: {e}'
    print(f"❌ Pipeline loading failed: {e}")

# Test 2: Load configurations
try:
    with open('flask_components/flask_config.json', 'r', encoding='utf-8') as f:
        test_config = json.load(f)
    verification_results['config_loading'] = '✅ PASS'
    print("✅ Flask config loaded successfully")
except Exception as e:
    verification_results['config_loading'] = f'❌ FAIL: {e}'
    print(f"❌ Config loading failed: {e}")

# Test 3: Test prediction function
try:
    sample_data = {
        'Year': 2024,
        'Month': 12,
        'Day': 15,
        'DayOfWeek': 0,
        'DayOfYear': 350,
        'Quarter': 4,
        'Temperatur': 26.0,
        'Curah_Hujan': 5.0,
        'Temperatur_rolling_7': 25.8,
        'Curah_Hujan_rolling_7': 4.5,
        'Curah_Hujan_rolling_30': 6.0,
        'Beras Kualitas Bawah I_lag1': 12350,
        'Beras Kualitas Bawah I_lag7': 12300,
        'Beras Kualitas Bawah I_lag30': 12200,
        'Beras Kualitas Bawah I_rolling_mean_7': 12325,
        'Beras Kualitas Bawah I_rolling_mean_30': 12250
    }
    
    pred_result = predict_price(sample_data, kategori_beras[0], test_pipeline)
    
    if pred_result['success']:
        verification_results['prediction_test'] = '✅ PASS'
        print(f"✅ Prediction test successful: Rp {pred_result['prediction']:,.0f}")
    else:
        verification_results['prediction_test'] = f"❌ FAIL: {pred_result.get('error', 'Unknown error')}"
        print(f"❌ Prediction test failed: {pred_result.get('error', 'Unknown error')}")
except Exception as e:
    verification_results['prediction_test'] = f'❌ FAIL: {e}'
    print(f"❌ Prediction test failed: {e}")

# Test 4: Check file completeness
required_files = {
    'models': ['complete_pipeline.pkl', 'svr_models.pkl', 'svr_scalers.pkl', 'prediction_functions.pkl'],
    'data': ['sorted_dataset.csv', 'processed_dataset.csv', 'split_data.pkl'],
    'results': ['final_performance_summary.json', 'error_analysis.json', 'svr_results.json']
}

missing_files = []
for folder, files in required_files.items():
    for file in files:
        filepath = f'flask_components/{folder}/{file}'
        if not os.path.exists(filepath):
            missing_files.append(f'{folder}/{file}')

if not missing_files:
    verification_results['file_completeness'] = '✅ PASS'
    print("✅ All required files present")
else:
    verification_results['file_completeness'] = f'❌ FAIL: Missing {missing_files}'
    print(f"❌ Missing files: {missing_files}")

# Final checklist
print("\n" + "="*80)
print("📋 FINAL PROJECT CHECKLIST")
print("="*80)

checklist_items = [
    "Dataset loaded dan dieksplorasi",
    "Visualisasi trend harga beras dengan Plotly",
    "Data preprocessing dan feature engineering (time, cuaca, lag, rolling)",
    "Data splitting chronological untuk time series",
    f"SVR model trained untuk {len(kategori_beras)} kategori beras",
    "Model evaluation dengan MAE, RMSE, MAPE metrics",
    "Tabel detail prediksi vs aktual untuk setiap kategori",
    "Visualisasi time series prediksi vs aktual",
    "Visualisasi model performance (scatter, residual)",
    "Error analysis per tahun dan kategori",
    "Model performance summary dan comparison",
    "Complete pipeline untuk Flask application",
    "Requirements.txt dan setup instructions",
    "Final research report dan documentation",
    "Verification testing passed"
]

print("COMPLETED TASKS:")
for i, item in enumerate(checklist_items, 1):
    print(f"☑️  {i:2d}. {item}")

# Summary statistics
print(f"\n📊 PROJECT STATISTICS:")
print("="*50)
print(f"Total records: {len(df):,} hari")
print(f"Processed records: {len(df_processed):,} hari")
print(f"Training samples: {split_idx:,}")
print(f"Testing samples: {len(df_processed) - split_idx:,}")
print(f"Categories trained: {len(kategori_beras)}")
print(f"Average MAPE: {avg_mape:.2f}%")
print(f"Average MAE: Rp {avg_mae:,.0f}")
print(f"Files created: {file_structure['total_files_created']}")

# Save verification results
verification_summary = {
    'verification_results': verification_results,
    'checklist_completed': len(checklist_items),
    'project_statistics': {
        'total_records': len(df),
        'processed_records': len(df_processed),
        'training_samples': split_idx,
        'testing_samples': len(df_processed) - split_idx,
        'categories_trained': len(kategori_beras),
        'average_mape': float(avg_mape),
        'average_mae': float(avg_mae),
        'files_created': file_structure['total_files_created']
    },
    'completion_timestamp': pd.Timestamp.now().isoformat()
}

with open('flask_components/verification_results.json', 'w', encoding='utf-8') as f:
    json.dump(verification_summary, f, indent=2)

print(f"\n🎊 CONGRATULATIONS!")
print("="*80)
print("Proyek prediksi harga bahan pangan Kabupaten Langkat sudah COMPLETE!")
print("Semua komponen siap untuk implementasi Flask.")
print("Happy coding! 🚀")

print(f"\n📁 FLASK COMPONENTS LOCATION: ./flask_components/")
print("   ✅ SVR Models saved (6 kategori)")
print("   ✅ Scalers saved")
print("   ✅ Data processed and saved")
print("   ✅ Results and metrics saved")
print("   ✅ Visualizations generated")
print("   ✅ Configuration files created")
print("   ✅ Final report completed")
print("   ✅ Verification passed")

print("\n✅ NOTEBOOK EXECUTION COMPLETED SUCCESSFULLY!")
print("💾 All components ready for Flask integration!")
print(f"\n📊 Metode: Support Vector Regression (SVR) dengan kernel RBF")
print(f"🎯 Evaluasi: MAE, RMSE, MAPE")
print(f"📈 Performa: MAPE rata-rata {avg_mape:.2f}%")


🧪 VERIFICATION - TESTING ALL COMPONENTS
✅ Complete pipeline loaded successfully
✅ Flask config loaded successfully
✅ Prediction test successful: Rp 11,156
✅ All required files present

📋 FINAL PROJECT CHECKLIST
COMPLETED TASKS:
☑️   1. Dataset loaded dan dieksplorasi
☑️   2. Visualisasi trend harga beras dengan Plotly
☑️   3. Data preprocessing dan feature engineering (time, cuaca, lag, rolling)
☑️   4. Data splitting chronological untuk time series
☑️   5. SVR model trained untuk 6 kategori beras
☑️   6. Model evaluation dengan MAE, RMSE, MAPE metrics
☑️   7. Tabel detail prediksi vs aktual untuk setiap kategori
☑️   8. Visualisasi time series prediksi vs aktual
☑️   9. Visualisasi model performance (scatter, residual)
☑️  10. Error analysis per tahun dan kategori
☑️  11. Model performance summary dan comparison
☑️  12. Complete pipeline untuk Flask application
☑️  13. Requirements.txt dan setup instructions
☑️  14. Final research report dan documentation
☑️  15. Verification testing

In [16]:
# Cell 16: Prediksi Masa Depan 2 Tahun Ke Depan (dengan Estimasi Cuaca Klimatologis) + Save

print("🔮 PREDIKSI HARGA BERAS 2 TAHUN KE DEPAN (730 HARI)")
print("=" * 80)

import datetime

# Konfigurasi prediksi
future_days = 730  # 2 tahun
last_date = df_processed['Tanggal'].max()
start_prediction_date = last_date + pd.Timedelta(days=1)

print(f"📅 Tanggal terakhir data: {last_date.strftime('%Y-%m-%d')}")
print(f"📅 Mulai prediksi dari: {start_prediction_date.strftime('%Y-%m-%d')}")
print(f"📅 Total hari prediksi: {future_days} hari (2 tahun)")

# Generate future dates
future_dates = pd.date_range(start=start_prediction_date, periods=future_days, freq='D')

print(f"📅 Rentang prediksi: {future_dates[0].strftime('%Y-%m-%d')} s/d {future_dates[-1].strftime('%Y-%m-%d')}")

# ===== KLIMATOLOGI CUACA UNTUK ESTIMASI MASA DEPAN =====
# Karena tidak ada data ramalan cuaca real untuk 2 tahun ke depan, dipakai
# pendekatan "normal klimatologis": rata-rata historis Temperatur & Curah Hujan
# per hari-dalam-tahun (day-of-year) dari 5 tahun data (2020-2024). Pendekatan
# ini umum dipakai ketika fitur eksogen masa depan tidak diketahui.
print(f"\n🌦️  MENGHITUNG KLIMATOLOGI CUACA (rata-rata historis per hari-dalam-tahun)...")
print("=" * 80)

climatology = df.groupby(df['Tanggal'].dt.dayofyear)[['Temperatur', 'Curah_Hujan']].mean()
climatology_dict = climatology.to_dict('index')

def get_climatology(date):
    """Ambil estimasi cuaca klimatologis untuk tanggal tertentu (handle hari ke-366 di tahun kabisat)."""
    doy = date.dayofyear
    if doy not in climatology_dict:
        doy = 365
    return climatology_dict[doy]['Temperatur'], climatology_dict[doy]['Curah_Hujan']

print(f"✅ Klimatologi dihitung untuk {len(climatology_dict)} hari-dalam-tahun")
print(f"   Contoh - Hari ke-1 (awal Januari): Temperatur={climatology_dict[1]['Temperatur']:.2f}°C, Curah Hujan={climatology_dict[1]['Curah_Hujan']:.2f}mm")
print(f"   Contoh - Hari ke-200 (pertengahan tahun): Temperatur={climatology_dict[200]['Temperatur']:.2f}°C, Curah Hujan={climatology_dict[200]['Curah_Hujan']:.2f}mm")

# Dictionary untuk menyimpan prediksi semua kategori
future_predictions = {}

# Untuk setiap kategori beras
for kategori in kategori_beras:
    print(f"\n{'='*80}")
    print(f"🔮 PREDIKSI UNTUK: {kategori}")
    print(f"{'='*80}")

    # Get model dan scaler
    model = svr_models[kategori]
    scaler_X = svr_scalers[kategori]['scaler_X']
    scaler_y = svr_scalers[kategori]['scaler_y']

    # Get last known data untuk lag features harga
    last_30_days = df_processed[[kategori]].tail(30).copy()

    # Get last known data cuaca aktual (untuk rolling cuaca di awal periode prediksi)
    last_30_weather = df_processed[['Temperatur', 'Curah_Hujan']].tail(30).copy()

    # Inisialisasi list untuk menyimpan prediksi
    predictions = []

    # Loop untuk setiap hari prediksi
    for day_idx in range(future_days):
        current_date = future_dates[day_idx]

        # Extract time features
        year = current_date.year
        month = current_date.month
        day = current_date.day
        dayofweek = current_date.dayofweek
        dayofyear = current_date.dayofyear
        quarter = current_date.quarter

        # Estimasi cuaca hari ini dari klimatologi
        temp_today, curah_today = get_climatology(current_date)

        # Ambil lag features dari data historis + prediksi sebelumnya
        if day_idx == 0:
            # Hari pertama, ambil dari data asli
            lag1 = last_30_days[kategori].iloc[-1]
            lag7 = last_30_days[kategori].iloc[-7] if len(last_30_days) >= 7 else last_30_days[kategori].iloc[0]
            lag30 = last_30_days[kategori].iloc[-30] if len(last_30_days) >= 30 else last_30_days[kategori].iloc[0]
            rolling_7 = last_30_days[kategori].tail(7).mean()
            rolling_30 = last_30_days[kategori].tail(30).mean()

            weather_combined_temp = last_30_weather['Temperatur']
            weather_combined_curah = last_30_weather['Curah_Hujan']
        else:
            # Hari berikutnya, gunakan kombinasi data asli + prediksi
            # Gabungkan data historis dengan prediksi yang sudah dibuat
            combined_data = pd.concat([
                last_30_days[kategori],
                pd.Series([p['predicted_price'] for p in predictions])
            ]).reset_index(drop=True)

            lag1 = combined_data.iloc[-1]
            lag7 = combined_data.iloc[-7] if len(combined_data) >= 7 else combined_data.iloc[0]
            lag30 = combined_data.iloc[-30] if len(combined_data) >= 30 else combined_data.iloc[0]
            rolling_7 = combined_data.tail(7).mean()
            rolling_30 = combined_data.tail(30).mean()

            weather_combined_temp = pd.concat([
                last_30_weather['Temperatur'],
                pd.Series([p['temperatur'] for p in predictions])
            ]).reset_index(drop=True)
            weather_combined_curah = pd.concat([
                last_30_weather['Curah_Hujan'],
                pd.Series([p['curah_hujan'] for p in predictions])
            ]).reset_index(drop=True)

        # Rolling cuaca dihitung dari kombinasi cuaca aktual + estimasi klimatologis,
        # konsisten dengan cara rolling harga dihitung secara rekursif
        temp_rolling_7 = pd.concat([weather_combined_temp, pd.Series([temp_today])]).tail(7).mean()
        curah_rolling_7 = pd.concat([weather_combined_curah, pd.Series([curah_today])]).tail(7).mean()
        curah_rolling_30 = pd.concat([weather_combined_curah, pd.Series([curah_today])]).tail(30).mean()

        # Buat feature vector
        features = {
            'Year': year,
            'Month': month,
            'Day': day,
            'DayOfWeek': dayofweek,
            'DayOfYear': dayofyear,
            'Quarter': quarter,
            'Temperatur': temp_today,
            'Curah_Hujan': curah_today,
            'Temperatur_rolling_7': temp_rolling_7,
            'Curah_Hujan_rolling_7': curah_rolling_7,
            'Curah_Hujan_rolling_30': curah_rolling_30,
            f'{kategori}_lag1': lag1,
            f'{kategori}_lag7': lag7,
            f'{kategori}_lag30': lag30,
            f'{kategori}_rolling_mean_7': rolling_7,
            f'{kategori}_rolling_mean_30': rolling_30
        }

        # Convert ke DataFrame
        feature_cols = [
            'Year', 'Month', 'Day', 'DayOfWeek', 'DayOfYear', 'Quarter',
            'Temperatur', 'Curah_Hujan', 'Temperatur_rolling_7', 'Curah_Hujan_rolling_7', 'Curah_Hujan_rolling_30',
            f'{kategori}_lag1', f'{kategori}_lag7', f'{kategori}_lag30',
            f'{kategori}_rolling_mean_7', f'{kategori}_rolling_mean_30'
        ]

        X_pred = pd.DataFrame([features])[feature_cols]

        # Scale features
        X_pred_scaled = scaler_X.transform(X_pred)

        # Prediksi
        y_pred_scaled = model.predict(X_pred_scaled)
        y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1))[0][0]

        # Simpan hasil prediksi
        predictions.append({
            'date': current_date,
            'predicted_price': y_pred,
            'year': year,
            'month': month,
            'day': day,
            'dayofweek': dayofweek,
            'dayofyear': dayofyear,
            'quarter': quarter,
            'temperatur': temp_today,
            'curah_hujan': curah_today
        })

        # Progress indicator setiap 100 hari
        if (day_idx + 1) % 100 == 0:
            print(f"  ✅ Progress: {day_idx + 1}/{future_days} hari ({(day_idx + 1)/future_days*100:.1f}%)")

    # Convert to DataFrame
    predictions_df = pd.DataFrame(predictions)

    # Statistik prediksi
    min_price = predictions_df['predicted_price'].min()
    max_price = predictions_df['predicted_price'].max()
    mean_price = predictions_df['predicted_price'].mean()
    std_price = predictions_df['predicted_price'].std()

    print(f"\n📊 STATISTIK PREDIKSI {kategori}:")
    print(f"  Min:  Rp {min_price:,.0f}")
    print(f"  Max:  Rp {max_price:,.0f}")
    print(f"  Mean: Rp {mean_price:,.0f}")
    print(f"  Std:  Rp {std_price:,.0f}")

    # Simpan ke dictionary
    future_predictions[kategori] = predictions_df

    # Save to CSV
    safe_kategori = kategori.replace(' ', '_').replace('/', '_')
    predictions_df.to_csv(f'flask_components/data/future_predictions_{safe_kategori}.csv', index=False)
    print(f"  💾 Saved: future_predictions_{safe_kategori}.csv")

print(f"\n{'='*80}")
print(f"✅ PREDIKSI SELESAI UNTUK SEMUA KATEGORI!")
print(f"{'='*80}")

# ==================== TAMPILKAN SAMPLE PREDIKSI ====================
print(f"\n📋 SAMPLE PREDIKSI (10 Hari Pertama & 10 Hari Terakhir)")
print("="*120)

for kategori in kategori_beras:
    predictions_df = future_predictions[kategori]

    print(f"\n🌾 {kategori}")
    print("-"*120)

    # Format untuk display
    display_df = predictions_df.copy()
    display_df['Tanggal'] = display_df['date'].dt.strftime('%d/%m/%Y')
    display_df['Harga Prediksi'] = display_df['predicted_price'].apply(lambda x: f"Rp {x:,.0f}")
    display_df['Temperatur (°C)'] = display_df['temperatur'].round(2)
    display_df['Curah Hujan (mm)'] = display_df['curah_hujan'].round(2)

    display_cols = ['Tanggal', 'Harga Prediksi', 'Temperatur (°C)', 'Curah Hujan (mm)', 'year', 'month', 'quarter']

    print("\n10 Hari PERTAMA:")
    display(display_df[display_cols].head(10))

    print(f"\n10 Hari TERAKHIR:")
    display(display_df[display_cols].tail(10))

# ==================== VISUALISASI PREDIKSI MASA DEPAN ====================
print(f"\n{'='*80}")
print(f"📊 MEMBUAT VISUALISASI PREDIKSI MASA DEPAN")
print(f"{'='*80}")

# 1. Line Chart Prediksi Semua Kategori
fig_future_all = go.Figure()

colors = ['#FF6B35', '#4ECDC4', '#45B7D1', '#F7DC6F', '#BB8FCE', '#85C1E2']

for i, kategori in enumerate(kategori_beras):
    predictions_df = future_predictions[kategori]

    fig_future_all.add_trace(go.Scatter(
        x=predictions_df['date'],
        y=predictions_df['predicted_price'],
        mode='lines',
        name=kategori,
        line=dict(width=2, color=colors[i % len(colors)]),
        hovertemplate='%{x|%Y-%m-%d}<br>Prediksi: Rp %{y:,.0f}<extra></extra>'
    ))

fig_future_all.update_layout(
    title=f"Prediksi Harga Beras 2 Tahun Ke Depan ({start_prediction_date.strftime('%Y-%m-%d')} s/d {future_dates[-1].strftime('%Y-%m-%d')})",
    xaxis_title="Tanggal",
    yaxis_title="Harga Prediksi (Rp/kg)",
    font=dict(size=12),
    width=1400,
    height=700,
    yaxis={'tickformat': ',.0f'},
    hovermode='x unified',
    legend=dict(
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02
    )
)

fig_future_all.show()
fig_future_all.write_html('flask_components/visualizations/future_predictions_all_categories.html')

# 2. Prediksi per Kategori (Individual)
for kategori in kategori_beras:
    predictions_df = future_predictions[kategori]

    fig_future_cat = go.Figure()

    fig_future_cat.add_trace(go.Scatter(
        x=predictions_df['date'],
        y=predictions_df['predicted_price'],
        mode='lines',
        name='Prediksi',
        line=dict(width=3, color='#E74C3C'),
        fill='tonexty',
        hovertemplate='%{x|%Y-%m-%d}<br>Harga: Rp %{y:,.0f}<extra></extra>'
    ))

    fig_future_cat.update_layout(
        title=f"Prediksi Harga {kategori} - 2 Tahun Ke Depan",
        xaxis_title="Tanggal",
        yaxis_title="Harga Prediksi (Rp/kg)",
        font=dict(size=12),
        width=1400,
        height=600,
        yaxis={'tickformat': ',.0f'},
        hovermode='x unified'
    )

    fig_future_cat.show()

    # Save
    safe_kategori = kategori.replace(' ', '_').replace('/', '_')
    fig_future_cat.write_html(f'flask_components/visualizations/future_prediction_{safe_kategori}.html')

# 3. Perbandingan Harga Rata-rata per Tahun (Future)
future_yearly = []

for kategori in kategori_beras:
    predictions_df = future_predictions[kategori]

    for year in predictions_df['year'].unique():
        year_data = predictions_df[predictions_df['year'] == year]
        avg_price = year_data['predicted_price'].mean()

        future_yearly.append({
            'Year': int(year),
            'Category': kategori.replace('Beras Kualitas ', ''),
            'Avg_Price': avg_price
        })

future_yearly_df = pd.DataFrame(future_yearly)

fig_future_yearly = px.bar(
    future_yearly_df,
    x='Year',
    y='Avg_Price',
    color='Category',
    barmode='group',
    title='Prediksi Rata-rata Harga Beras per Tahun (Future)',
    labels={'Avg_Price': 'Harga Rata-rata (Rp/kg)', 'Year': 'Tahun'},
    color_discrete_sequence=colors
)

fig_future_yearly.update_layout(
    width=1200,
    height=600,
    font=dict(size=12),
    yaxis={'tickformat': ',.0f'}
)

fig_future_yearly.show()
fig_future_yearly.write_html('flask_components/visualizations/future_predictions_yearly_avg.html')

# 4. Distribusi Harga Prediksi per Kategori (Box Plot)
fig_future_box = go.Figure()

for i, kategori in enumerate(kategori_beras):
    predictions_df = future_predictions[kategori]

    fig_future_box.add_trace(go.Box(
        y=predictions_df['predicted_price'],
        name=kategori.replace('Beras Kualitas ', ''),
        marker_color=colors[i % len(colors)],
        boxmean='sd'
    ))

fig_future_box.update_layout(
    title="Distribusi Prediksi Harga Beras (2 Tahun Ke Depan)",
    yaxis_title="Harga (Rp/kg)",
    font=dict(size=12),
    width=1200,
    height=600,
    yaxis={'tickformat': ',.0f'},
    showlegend=True
)

fig_future_box.show()
fig_future_box.write_html('flask_components/visualizations/future_predictions_distribution.html')

print("\n📊 Visualisasi prediksi masa depan berhasil dibuat!")
print("💾 Semua visualisasi sudah disimpan!")

# ==================== SAVE SUMMARY ====================
future_summary = {
    'prediction_info': {
        'total_days': future_days,
        'start_date': start_prediction_date.strftime('%Y-%m-%d'),
        'end_date': future_dates[-1].strftime('%Y-%m-%d'),
        'last_historical_date': last_date.strftime('%Y-%m-%d'),
        'categories': len(kategori_beras),
        'weather_estimation_method': 'Climatological normal (rata-rata historis per hari-dalam-tahun, 2020-2024)'
    },
    'predictions_by_category': {}
}

for kategori in kategori_beras:
    predictions_df = future_predictions[kategori]

    future_summary['predictions_by_category'][kategori] = {
        'min_price': float(predictions_df['predicted_price'].min()),
        'max_price': float(predictions_df['predicted_price'].max()),
        'mean_price': float(predictions_df['predicted_price'].mean()),
        'std_price': float(predictions_df['predicted_price'].std()),
        'total_predictions': int(len(predictions_df))
    }

with open('flask_components/results/future_predictions_summary.json', 'w', encoding='utf-8') as f:
    json.dump(future_summary, f, indent=2, ensure_ascii=False)

print("\n✅ PREDIKSI MASA DEPAN SELESAI!")
print("💾 Semua data dan visualisasi sudah disimpan!")
print(f"\n📁 Files tersimpan di:")
print(f"   - Data CSV: flask_components/data/future_predictions_*.csv")
print(f"   - Visualisasi: flask_components/visualizations/future_prediction*.html")
print(f"   - Summary: flask_components/results/future_predictions_summary.json")

# ==================== TABEL PERBANDINGAN: HISTORICAL vs FUTURE ====================
print(f"\n{'='*120}")
print(f"📊 PERBANDINGAN: HARGA HISTORICAL vs PREDIKSI FUTURE")
print(f"{'='*120}")

comparison_data = []

for kategori in kategori_beras:
    # Historical stats (dari data asli)
    hist_mean = df[kategori].mean()
    hist_min = df[kategori].min()
    hist_max = df[kategori].max()

    # Future stats
    predictions_df = future_predictions[kategori]
    future_mean = predictions_df['predicted_price'].mean()
    future_min = predictions_df['predicted_price'].min()
    future_max = predictions_df['predicted_price'].max()

    # Perubahan
    change = future_mean - hist_mean
    change_pct = (change / hist_mean) * 100

    comparison_data.append({
        'Kategori': kategori.replace('Beras Kualitas ', ''),
        'Historical Mean': f"Rp {hist_mean:,.0f}",
        'Future Mean': f"Rp {future_mean:,.0f}",
        'Perubahan': f"Rp {change:,.0f}",
        'Perubahan (%)': f"{change_pct:+.2f}%",
        'Trend': '📈 Naik' if change > 0 else '📉 Turun' if change < 0 else '➡️ Stabil'
    })

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

# Save comparison
comparison_df.to_csv('flask_components/results/historical_vs_future_comparison.csv', index=False)

print(f"\n💾 Comparison table saved!")
print(f"\n🎊 PREDIKSI 2 TAHUN KE DEPAN COMPLETE!")


🔮 PREDIKSI HARGA BERAS 2 TAHUN KE DEPAN (730 HARI)
📅 Tanggal terakhir data: 2024-12-31
📅 Mulai prediksi dari: 2025-01-01
📅 Total hari prediksi: 730 hari (2 tahun)
📅 Rentang prediksi: 2025-01-01 s/d 2026-12-31

🌦️  MENGHITUNG KLIMATOLOGI CUACA (rata-rata historis per hari-dalam-tahun)...
✅ Klimatologi dihitung untuk 366 hari-dalam-tahun
   Contoh - Hari ke-1 (awal Januari): Temperatur=25.59°C, Curah Hujan=8.90mm
   Contoh - Hari ke-200 (pertengahan tahun): Temperatur=26.56°C, Curah Hujan=6.85mm

🔮 PREDIKSI UNTUK: Beras Kualitas Bawah I


  ✅ Progress: 100/730 hari (13.7%)


  ✅ Progress: 200/730 hari (27.4%)


  ✅ Progress: 300/730 hari (41.1%)


  ✅ Progress: 400/730 hari (54.8%)


...

,Tanggal,Harga Prediksi,Temperatur (°C),Curah Hujan (mm),year,month,quarter
720,22/12/2026,"Rp 13,492",24.74,17.65,2026,12,4
721,23/12/2026,"Rp 13,509",24.75,15.53,2026,12,4
722,24/12/2026,"Rp 13,482",25.43,8.57,2026,12,4
723,25/12/2026,"Rp 13,528",24.62,31.80,2026,12,4
724,26/12/2026,"Rp 13,433",24.71,9.75,2026,12,4
725,27/12/2026,"Rp 13,369",25.25,10.15,2026,12,4
726,28/12/2026,"Rp 13,394",25.40,4.32,2026,12,4
727,29/12/2026,"Rp 13,496",24.87,16.08,2026,12,4
728,30/12/2026,"Rp 13,468",24.83,5.73,2026,12,4
729,31/12/2026,"Rp 13,487",24.44,15.23,2026,12,4



📊 MEMBUAT VISUALISASI PREDIKSI MASA DEPAN



📊 Visualisasi prediksi masa depan berhasil dibuat!
💾 Semua visualisasi sudah disimpan!

✅ PREDIKSI MASA DEPAN SELESAI!
💾 Semua data dan visualisasi sudah disimpan!

📁 Files tersimpan di:
   - Data CSV: flask_components/data/future_predictions_*.csv
   - Visualisasi: flask_components/visualizations/future_prediction*.html
   - Summary: flask_components/results/future_predictions_summary.json

📊 PERBANDINGAN: HARGA HISTORICAL vs PREDIKSI FUTURE


,Kategori,Historical Mean,Future Mean,Perubahan,Perubahan (%),Trend
0,Bawah I,"Rp 10,618","Rp 11,198",Rp 579,+5.46%,📈 Naik
1,Bawah II,"Rp 11,103","Rp 11,676",Rp 572,+5.16%,📈 Naik
2,Medium I,"Rp 12,134","Rp 12,245",Rp 111,+0.92%,📈 Naik
3,Medium II,"Rp 12,306","Rp 12,859",Rp 553,+4.49%,📈 Naik
4,Super I,"Rp 12,988","Rp 13,088",Rp 100,+0.77%,📈 Naik
5,Super II,"Rp 13,135","Rp 13,395",Rp 259,+1.98%,📈 Naik



💾 Comparison table saved!

🎊 PREDIKSI 2 TAHUN KE DEPAN COMPLETE!
